# Coherence-Rate Graphs for Noisy Equivariant Quantum Neural Networks

This notebook is the complete experimental pipeline behind the paper
*A Coherence-Rate Diagnostic for Trainability Under Noise in Equivariant
Quantum Neural Networks*.
Executed top to bottom, it reproduces every numerical result in the paper
on a single CPU runtime.

The object of study is the **coherence-rate graph**. For the noise-channel
families treated here, the restricted generator acts diagonally on the
coherence pairs of a charge sector of a U(1)-equivariant circuit, and its
pairwise decay rates define a weighted graph on the sector basis. Questions
about noisy trainability then become questions about this graph, and the
notebook validates that reading end to end:

* **analytic channel rates** — closed-form pairwise decay rates for a
  reference suite of nine channels, with structural form certificates
  (difference form for the dephasing family, potential form for the
  damping family), kernels and protection components;
* **combinatorial certificates** — a cut-crossing predictor of the depth
  at which each trainable parameter becomes exposed to a correlated
  channel, and a path-support certificate whose satisfaction implies exact
  gradient invariance at every noise strength, both computed by Boolean
  reachability with no linear algebra;
* **response diagnostics** — a support bound, an adversarial sweep over
  random covariant generators, exact per-slot product laws for the
  measured degradation, difference-based equivalence analysis of the
  deep-circuit comparisons, and a deterministic prior-integrated
  population benchmark that supplies exact targets with no Monte Carlo.

**Modes.** `CGL_MODE` selects the run: `paper` (full ensembles, the
committed results), `smoke` (identical code paths at reduced ensembles,
minutes on a laptop; statistical columns indicative only) and `figures`
(read-only; renders from existing CSVs and never computes a phase).
Publication figures are built in the companion figures notebook of this
repository, which reads the cached CSVs only.

**Caching and provenance.** Every phase writes its CSV to the results
directory together with a `.meta.json` sidecar recording the mode, the
configuration fingerprint, all seeds and the hash of the phase source that
produced it. On re-run, a phase whose output and sidecar match is skipped,
and a cached output produced by a different phase implementation is
recomputed rather than silently reused. The four supporting modules are
written out exactly as executed, and the configuration fingerprint covers
their source hashes, so the committed CSVs are bound to the committed
code.


## Definitions used

**Model.** The U(1)-equivariant brickwork ansatz on the cycle $C_n$
($n = 8$, primary depth $L = 3$): trainable $R_z(\theta_{\ell q})$ on every
site, fixed seeded XY hopping on alternating edge sets $E_\ell$, readout
$P_r Z_0 P_r$, single Markovian noise slot after each layer. Charge sector
$r$ (primary $r = 1$; Phase G uses $r = 2$).

**Restricted generator.** $\mathcal{L}_r = (\Phi_{\gamma_p} - \mathrm{id})/
\gamma_p$ on the in-sector coherence-pair basis $\{|a\rangle\langle b|\}$,
at probe strength $\gamma_p = 10^{-3}$ (carrying the documented
$O(\gamma_p)$ bias).

**Coherence-rate graph.** Pair-diagonality is a property of stated channel
families, **not** a consequence of phase covariance (a covariant site-swap
generator mixes pairs): the channels built from computational-basis-diagonal
jump operators, and in-sector damping, act diagonally on coherence pairs at
the infinitesimal level, and for those families the pairwise decay rate
defines a weighted graph on the sector basis states. The **difference
form** holds for the dephasing family:
$\mathrm{rate}(a,b) = \lVert v(a) - v(b)\rVert^2$ with the jump-spectrum
embedding $v_k(a) = \sqrt{c_k/2}\,\ell_k(a)$ under the Lindblad
normalisation $\mathcal{D}(X)=\sum_k c_k (J_k X J_k^\dagger - \tfrac12
\{J_k^\dagger J_k, X\})$; probabilistic-unitary channel families carry
their own stated coefficients (Table conventions in Phase B). The
**potential (Schrödinger) form** holds for the damping family:
$\mathrm{rate}(a,b) = \varphi(a) + \varphi(b)$ with
$\varphi(a) = \tfrac12 \sum_{q \in \mathrm{exc}(a)} w_q$.

**Three distinct objects.** For a pair-diagonal channel: (i) the
**pair-space decay operator** $R = \mathrm{diag}\{r(a,b)\}$
on the $d_r(d_r-1)$-dimensional ordered-pair space, whose kernel dimension
counts zero-rate ordered pairs and which governs coherence attenuation;
(ii) the **vertex-weight Laplacian** $L_V = \mathrm{diag}(W\mathbf{1}) - W$
with $W_{ab} = r(a,b)$ on the $d_r$-dimensional vertex space, an analysis
object whose spectrum is generally unrelated to $R$ (for the correlated
control at $n=8$, $r=1$ it is connected with a single zero eigenvalue while
$R$ has an eight-dimensional kernel); and (iii) the **protection graph**,
the zero-rate edge set, whose components organise the protected pairs. Each
result below names the object it concerns. Pair-diagonality holds for the
infinitesimal generator of the stated families; the finite-probe object of
Eq.\,(1) can acquire $O(\gamma_p)$ mixing, whose scope by channel and
sector Phase I quantifies.

**Protection graph and kernel.** Pairs with zero rate form the protection
edge set; its ordered-pair count equals the kernel dimension of the
Hermitian part of $-\mathcal{L}_r$, and its connected components organise
the protected subspace.

**Cut-crossing predictor.** Per noise slot $\ell$, the visible interaction
cone $I(\ell)$ is the intersection of the readout's backward cone (through
the functional side) with the state's forward cone, computed by pure
edge-set reachability. A slot is combinatorially protected when every
2-subset of $I(\ell)$ is a protected pair. All slots protected is a
*sufficient* condition for zero response; predicted exposure is a
*necessary* condition only, since cancellations can still protect.

**Mode and response rates.** $\lambda_{\mathrm{coh}}$ (worst case),
$\lambda_{\mathrm{mode}}$ (ensemble Rayleigh quotient of the gradient
mode) and the response susceptibility $\Lambda^{\mathrm{resp}} = \sum_\ell
r_\ell$ with $\omega = \Lambda^{\mathrm{resp}}/(L\,\lambda_{\mathrm{coh}})$
are the objects of the supporting rate module, used unchanged throughout.

**Support bound.** (A weighted-average inequality; no cut conductance or spectral gap enters, so it is not described as Cheeger-type.)
 For a diagonal generator,
$\lambda_{\mathrm{mode}} = \sum_p \mathrm{rate}_p\, |g_p|^2 / \sum_p
|g_p|^2 \ \ge\ \phi\, r_{\min}$, where $\phi$ is the mode's weight fraction
on unprotected pairs and $r_{\min}$ the smallest nonzero unprotected rate.

**Exact slot product laws.** With independent per-slot strengths
$g_\ell$, the measured ratio at the audited shallow geometry obeys exact
product laws, $M_2(\mathbf{g})/M_2(0) = \prod_\ell (1 - \alpha
g_\ell)^{k_\ell}$, with integer exponents derived from the contributing
matrix-unit paths (Phase F2D); uniform strength gives closed-form
$\Lambda_1 = \alpha K/2$ and $K_2 = \alpha^2 K(K-1)/2$ with
$K = \sum_\ell k_\ell$. For the implemented probability
parameterisation the same-slot second derivative is **not** the squared
generator (a damping slot is affine in $\gamma$ in-sector), so no
biharmonic reading is attached; the laws are exact statements, not
low-order approximations.


## Phase guide

| Phase | Question answered |
|---|---|
| A | Do all exact identities hold (trace, sector, diagonality, analytic rates, structural forms, kernel, slotwise consistency)? |
| B | Is the coherence-rate graph recovered a priori for every channel: rates, spectra, kernels, components, form certificates? |
| C | Does the combinatorial cut-crossing predictor reproduce the depth at which the correlated control's protection switches off, and does the visible cut mass track the measured exposure? |
| D | Does the support bound hold across channels and parameters, and does the mode-response ordering survive an adversarial sweep over random covariant generators? |
| E | Are the closed-form slot targets (2/3, 1/2, final slot zero), recorded in the source before measurement, reproduced? |
| G | How does the protection structure collapse in the two-excitation sector, seen as graph spectra? |
| H | What do high-ensemble deep-circuit measurements give at depths five and six, with a three-strength extrapolation to the first-order limit? |
| I | Where does pair-diagonality hold: infinitesimal generator versus the finite-probe object, by channel and sector? |
| F2 | Do the measured ratios obey exact per-slot product laws, with closed-form coefficients to which finite differences converge? |
| J | Is shallow protection a property of the readout-visible pairing rather than the full mode, and does the certificate require dissipation? |
| K | How does the visible response weight overlap the protected subspace at $r=2$? |
| H2 | Under a difference-based equivalence analysis with replication, is the depth-six comparison equivalent, different, or inconclusive? |
| F2D | Do measurements match slot exponents derived before testing? |
| J2 | Is the all-orders protection an exact fixed-point theorem with stated hypotheses? |
| K2 | With sector and input state separated in a full factorial, which factor breaks protection? |
| J3 | Does the purely combinatorial path-support certificate hold, and does it imply joint all-strength invariance? |
| P | What are the exact population response values, and how do the estimators and extrapolations sit against them? |

Each phase writes one or two CSVs to the results directory and is cached.
Assertions are confined to Phase A's exact identities; later phases record
pass/fail columns so that reduced smoke ensembles cannot abort the run on
statistical fluctuations. Phase H supplies raw deep-circuit measurements;
the decision rule applied to the depth-six question is the difference-based
rule of Phase H2.


In [ ]:
# ============================== Configuration ==============================
import os, sys, json, time, platform
from pathlib import Path
import numpy as np
import pandas as pd

PIPELINE_VERSION = "1.5"
MODE = os.environ.get("CGL_MODE", "paper")           # "smoke" | "paper" | "figures"
FORCE_RERUN = False
ALLOW_LEGACY_CACHE = True        # reuse pre-sidecar / older-fingerprint caches
ALLOW_MISMATCHED_CACHE = False   # NEVER silently reuse a same-version mismatch
assert MODE in ("smoke", "paper", "figures")
FIGURES_ONLY = (MODE == "figures")

# ---- output location: Google Drive when on Colab, local folder otherwise ----
GDRIVE_RESULTS = ("Colab Notebooks/.../")
try:                                   # running on Google Colab?
    from google.colab import drive     # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd()
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive") / GDRIVE_RESULTS
else:
    OUT_DIR = ROOT / "cgl_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures";         FIG_DIR.mkdir(exist_ok=True)
PKG_DIR = ROOT / "cgl";                PKG_DIR.mkdir(exist_ok=True)

if MODE == "paper":
    CFG = dict(
        n=8, L=3, r=1,                        # primary configuration
        depths=(3, 4, 5, 6),                  # Phase C depth sweep
        gamma_meas=(0.01, 0.02),              # per-layer strengths, Phase C
        so2_gl_grid=(0.01, 0.02, 0.035, 0.05, 0.10, 0.20, 0.30),   # Phase F noise grid
        so2_fd_small=(1e-3, 2e-3),            # Phase F first-order slope
        hess_h=1e-2,                          # Phase F Hessian step
        n_theta_est=40,                       # estimator ensemble size
        n_theta_meas=30,                      # CRN draws per degradation point
        n_theta_hess=20,                      # shared draws, Phase F
        preflight_draws=8,
        bootstrap_B=1000,
        sweep_profiles=150,                   # random dephasing profiles
        sweep_edge_sets=50,                   # random ZZ edge-set channels
        r2_numeric_channels=("dephase", "inhom_dephase", "corr_dephase",
                             "amp_damp", "site_amp_damp", "depol"),
        stress_depths=(5, 6),                 # Phase H: control, stress
        stress_est=120,                       # estimator draws, Phase H
        stress_meas=400,                      # CRN draws per point, Phase H
        gamma_stress=(0.005, 0.01, 0.02),     # three-point extrapolation
        # ---- v1.2 additions (review-driven) ----
        r2_offdiag_channels=("depol", "x_err", "biased_pauli",
                             "dephase", "corr_dephase", "amp_damp"),
        offdiag_probes=(1e-3, 5e-4),
        f2_slot_strengths=(0.10, 0.23),
        f2_product_draws=5,
        f2_fd_steps=(1e-2, 5e-3, 2.5e-3, 1.25e-3),
        j_gammas=(0.10, 0.30),
        k_meas=30,
        eq_margin=0.01,
        h2_sizes=(40, 120),
        h2_reps=10,
    )
else:
    CFG = dict(
        n=8, L=3, r=1,
        depths=(3, 4),
        gamma_meas=(0.02,),
        so2_gl_grid=(0.01, 0.02, 0.035, 0.05, 0.10, 0.20),
        so2_fd_small=(1e-3, 2e-3),
        hess_h=1e-2,
        n_theta_est=6,
        n_theta_meas=8,
        n_theta_hess=6,
        preflight_draws=3,
        bootstrap_B=100,
        sweep_profiles=8,
        sweep_edge_sets=4,
        r2_numeric_channels=("dephase", "corr_dephase"),
        stress_depths=(5, 6),
        stress_est=8,
        stress_meas=8,
        gamma_stress=(0.005, 0.01, 0.02),
        r2_offdiag_channels=("depol", "x_err", "dephase"),
        offdiag_probes=(1e-3, 5e-4),
        f2_slot_strengths=(0.10, 0.23),
        f2_product_draws=3,
        f2_fd_steps=(1e-2, 5e-3),
        j_gammas=(0.10, 0.30),
        k_meas=8,
        eq_margin=0.01,
        h2_sizes=(6, 12),
        h2_reps=3,
    )

DELTA_INIT   = 0.05          # small-box prior half-width
GAMMA_PROBE  = 1e-3          # generator probe strength
SHIFT        = np.pi / 2     # parameter-shift value
TEACHER_SEED = 42            # hopping-background seed (CohAlign convention)
PREFLIGHT_SEED, EST_SEED, MEAS_SEED, BOOT_SEED = 11, 1, 1234, 7
SWEEP_SEED, HESS_SEED = 33, 55

TIMINGS = {}

import hashlib as _hl
FINGERPRINT_VERSION = 3
def _cfg_fingerprint():
    """v2 fingerprint: mode, full configuration, every seed, probe strength,
    prior half-width, parameter-shift value, and the SHA-256 of each module
    source present in the session (so an edited implementation cannot
    silently consume old outputs)."""
    blob = json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in CFG.items()}, sort_keys=True)
    extras = {"seeds": [TEACHER_SEED, PREFLIGHT_SEED, EST_SEED, MEAS_SEED,
                        BOOT_SEED, SWEEP_SEED, HESS_SEED],
              "gamma_probe": GAMMA_PROBE, "delta_init": DELTA_INIT,
              "shift": float(SHIFT)}
    mods = []
    for nm in ("COHALIGN_CORE_SRC", "COHALIGN_RATES_SRC",
               "COHALIGN_BENCH_SRC", "CGL_GRAPH_SRC"):
        src = globals().get(nm)
        if src is not None:
            mods.append(_hl.sha256(src.encode()).hexdigest()[:12])
    payload = (MODE + blob + json.dumps(extras, sort_keys=True)
               + "".join(mods))
    return _hl.sha256(payload.encode()).hexdigest()[:16]

def _executing_cell_hash():
    """SHA-256 (12 hex) of the source of the cell currently executing.
    This binds each cache to the code that actually ran, replacing the
    v1.4 assembly-time registry, which the audit showed could disagree
    with the executed source (and did, for three shared-fragment phases).
    Returns None when the interactive source is unavailable."""
    try:
        ip = get_ipython()                      # noqa: F821
        src = ip.user_ns.get("In", [None])[-1]
        if src:
            return _hl.sha256(src.encode()).hexdigest()[:12]
    except Exception:
        pass
    return None

def _csv_snapshot():
    return {p.name: _hl.sha256(p.read_bytes()).hexdigest()
            for p in sorted(OUT_DIR.glob("*.csv"))}
if FIGURES_ONLY:
    _FIGURES_SNAPSHOT = _csv_snapshot()

def cache_or_compute(path, label, inputs=None):
    """Existence check plus a provenance sidecar audit: a cached CSV whose
    .meta.json is missing, or was written under a different mode or
    configuration fingerprint, is still reused (so cross-mode reuse remains
    an explicit choice) but a visible warning is printed. write_meta() must
    be called by every phase after writing its CSV."""
    if FIGURES_ONLY:
        # figures-only mode is READ-ONLY by contract: it renders from
        # whatever numerical artefacts exist and never computes a phase,
        # regardless of any provenance mismatch (v1.4, audit repair)
        if path.exists():
            print(f"[figures] {label}: rendering from existing {path.name}")
            return False
        raise FileNotFoundError(
            f"{label}: {path.name} missing but MODE='figures'. "
            "Run smoke or paper mode first.")
    if path.exists() and not FORCE_RERUN:
        meta_p = path.with_suffix(path.suffix + ".meta.json")
        if meta_p.exists():
            try:
                m = json.loads(meta_p.read_text())
            except Exception:
                m = None
            if m is None:
                if not ALLOW_LEGACY_CACHE:
                    print(f"[cache] {label}: unreadable sidecar, recomputing")
                    return True
                print(f"[cache] {label}: using existing {path.name}"
                      "  [note] unreadable sidecar, reused as legacy")
                return False
            if m.get("backfilled"):
                # a backfilled record never becomes execution provenance
                if not ALLOW_LEGACY_CACHE:
                    print(f"[cache] {label}: backfilled provenance refused, "
                          "recomputing")
                    return True
                print(f"[cache] {label}: using existing {path.name}"
                      "  [note] backfilled provenance, reused as legacy")
                return False
            # source-to-cache binding: if the sidecar records the phase
            # source that produced it and the currently executing cell
            # differs, the cache is stale regardless of its age
            rec_src = m.get("phase_source_sha256")
            cur_src = _executing_cell_hash()
            if (rec_src is not None and cur_src is not None
                    and rec_src != cur_src and not ALLOW_MISMATCHED_CACHE):
                print(f"[cache] {label}: cached output was produced by a "
                      "DIFFERENT phase implementation; recomputing")
                return True
            # strict mode treats MISSING source evidence on a
            # current-version record as unverified, not as verified
            if (rec_src is None and cur_src is not None
                    and m.get("fingerprint_version") == FINGERPRINT_VERSION
                    and not ALLOW_MISMATCHED_CACHE):
                print(f"[cache] {label}: current-version record lacks "
                      "source evidence; recomputing")
                return True
            # upstream-input binding: a cache is stale when the inputs it
            # summarised have changed, even though its own bytes verify
            rec_in = m.get("input_sha256", {})
            for ip_ in (inputs or []):
                ip_ = Path(ip_)
                if not ip_.exists():
                    print(f"[cache] {label}: upstream input {ip_.name} "
                          "missing; recomputing")
                    return True
                cur = _hl.sha256(ip_.read_bytes()).hexdigest()
                if ip_.name not in rec_in:
                    print(f"[cache] {label}: no recorded provenance for "
                          f"upstream input {ip_.name}; recomputing")
                    return True
                if rec_in[ip_.name] != cur:
                    print(f"[cache] {label}: upstream input {ip_.name} "
                          "has CHANGED since this cache was written; "
                          "recomputing")
                    return True
            stored_hash = m.get("output_sha256")
            if stored_hash is not None:
                actual = _hl.sha256(path.read_bytes()).hexdigest()
                if actual != stored_hash:
                    print(f"[cache] {label}: OUTPUT CONTENT does not match "
                          "its provenance record; recomputing")
                    return True
            comp_hashes = m.get("companion_sha256", {})
            for comp in m.get("companions", []):
                cp = OUT_DIR / comp
                if not cp.exists():
                    print(f"[cache] {label}: companion {comp} missing; "
                          "recomputing")
                    return True
                if comp in comp_hashes and _hl.sha256(
                        cp.read_bytes()).hexdigest() != comp_hashes[comp]:
                    print(f"[cache] {label}: companion {comp} content "
                          "does not match its provenance record; "
                          "recomputing")
                    return True
            same_ver = m.get("fingerprint_version") == FINGERPRINT_VERSION
            match = (m.get("mode") == MODE and
                     m.get("config_fingerprint") == _cfg_fingerprint())
            if same_ver and not match and not ALLOW_MISMATCHED_CACHE:
                print(f"[cache] {label}: cached under a different mode or "
                      "configuration (same fingerprint version); "
                      "RECOMPUTING (set ALLOW_MISMATCHED_CACHE=True to "
                      "reuse deliberately)")
                return True
            note = "" if (same_ver and match) else                 "  [note] older-fingerprint cache reused as legacy"
            if not same_ver and not ALLOW_LEGACY_CACHE:
                print(f"[cache] {label}: legacy cache refused, recomputing")
                return True
            print(f"[cache] {label}: using existing {path.name}{note}")
            return False
        if not ALLOW_LEGACY_CACHE:
            print(f"[cache] {label}: no sidecar, legacy reuse disabled, "
                  "recomputing")
            return True
        print(f"[cache] {label}: using existing {path.name}"
              "  [note] no provenance sidecar (pre-v1.2 cache)")
        return False
    if FIGURES_ONLY:
        raise FileNotFoundError(
            f"{label}: {path.name} missing but MODE='figures'. "
            "Run smoke or paper mode first.")
    return True

def write_meta(path, label, extra=None, companions=None, inputs=None):
    meta = {"phase": label, "mode": MODE,
            "fingerprint_version": FINGERPRINT_VERSION,
            "config_fingerprint": _cfg_fingerprint(),
            "phase_source_sha256": _executing_cell_hash(),
            "output_sha256":
                _hl.sha256(path.read_bytes()).hexdigest(),
            "companions": companions or [],
            "companion_sha256":
                {c: _hl.sha256((OUT_DIR / c).read_bytes()).hexdigest()
                 for c in (companions or []) if (OUT_DIR / c).exists()},
            "input_sha256":
                {Path(i).name: _hl.sha256(Path(i).read_bytes()).hexdigest()
                 for i in (inputs or []) if Path(i).exists()},
            "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "seeds": {"teacher": TEACHER_SEED, "preflight": PREFLIGHT_SEED,
                      "est": EST_SEED, "meas": MEAS_SEED, "boot": BOOT_SEED,
                      "sweep": SWEEP_SEED, "hess": HESS_SEED}}
    if extra:
        meta.update(extra)
    path.with_suffix(path.suffix + ".meta.json").write_text(
        json.dumps(meta, indent=2))

# ---- v1.2 analysis plan, written before any phase executes ----
PLAN = OUT_DIR / "ANALYSIS_PLAN_v1_2.md"
if not PLAN.exists():
    PLAN.write_text("""# Analysis plan, pipeline v1.2 (recorded before execution)

Phase F2 predictions (exact, replacing the v1.1 finite-difference fit):
the measured second-moment ratio at the audited depth-three geometry obeys
an exact per-slot product law, dephasing (1-2g_l)^4 on each of the two
visible slots and amplitude damping (1-g_l)^2 on all three slots, giving
Lambda1 = 8, K2 = 112 (dephasing) and Lambda1 = 3, K2 = 15 (damping),
with the finite-step Hessian of v1.1 expected to converge to these values
linearly in the step size.

Phase J prediction: correlated-control parameters whose visible pair
weight is entirely on protected pairs have measured ratio identically 1
at every strength, even where the full gradient mode decays
(lambda_mode > 0); the pure coherent site-Z control has zero Hermitian
rate but a nonzero measured response, so the protection certificate
carries a dissipative hypothesis.

Phase H2 decision rule: the depth-six question is decided on the
DIFFERENCE between the extrapolated measurement and the prediction, both
resampled at draw level. Equivalent if the 90% difference interval lies
inside +/-0.01; different if the 95% interval excludes +/-0.01 entirely;
inconclusive otherwise. The depth-five control must be equivalent for
either stress verdict to stand. The replication study (independent seed
replicates at two estimator sizes) decides between estimator bias and
sampling fluctuation; bias language is used only if the size-40 and
size-120 replicate means differ by more than their combined spread.

External timestamping of this plan (repository commit or preprint) is a
manual step outside the notebook.
""")
    print(f"analysis plan written -> {PLAN.name}")

t_notebook_start = time.time()
print(f"Coherence-graph pipeline | MODE={MODE} | "
      f"n={CFG['n']} L={CFG['L']} r={CFG['r']}")
print(f"environment: {'Google Colab (Drive-mounted)' if IN_COLAB else 'local'}")
print(f"outputs -> {OUT_DIR}")

## Module 1 — `cohalign_core`

Supporting library: sector bookkeeping, the equivariant brickwork ansatz,
and the nine-channel noise suite.


In [ ]:
COHALIGN_CORE_SRC = r'''"""
cohalign_core.py -- model and noise-channel library for CohAlign.

Standalone implementation of:
  - charge-sector bookkeeping for the U(1) symmetry on n qubits
  - the U(1)-equivariant brickwork ansatz on the cycle C_n
    (trainable Rz layer followed by fixed XY hopping on alternating edges,
     single-qubit Markovian noise applied once per layer)
  - nine reference noise channels: four restricted-isotropic
    (amplitude damping, dephasing, depolarising, X-error) and five
    structured (inhomogeneous dephasing, site-dependent amplitude damping,
    biased Pauli, coherent-dissipative mix, correlated two-site dephasing)

Every channel exposes  apply(M, gamma, n)  acting on an arbitrary
2^n x 2^n matrix, so the same code path serves density-matrix evolution
and the linear-map probes used by cohalign_rates.
"""
import numpy as np
from itertools import combinations

# ---------------------------------------------------------------- Pauli
I2 = np.eye(2, dtype=complex)
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

# ------------------------------------------------------ sector bookkeeping
def sector_basis(n: int, r: int) -> list:
    """Computational-basis indices of Hamming weight r (bit q = site q)."""
    out = []
    for pos in combinations(range(n), r):
        s = 0
        for p in pos:
            s |= (1 << p)
        out.append(s)
    return sorted(out)

def sector_projector(n: int, r: int) -> np.ndarray:
    P = np.zeros((2**n, 2**n), dtype=complex)
    for s in sector_basis(n, r):
        P[s, s] = 1.0
    return P

def pair_basis(n: int, r: int) -> list:
    """Ordered pairs (a, b), a != b, spanning the in-sector off-diagonal block."""
    sec = sector_basis(n, r)
    return [(a, b) for a in sec for b in sec if a != b]

# ---------------------------------------------------------- embeddings
def embed_one_qubit(op: np.ndarray, q: int, n: int) -> np.ndarray:
    """Embed a 2x2 operator on qubit q (bit q of the integer index)."""
    mats = [I2] * n
    mats[n - 1 - q] = op
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out

def xy_unitary(beta: float, i: int, j: int, n: int) -> np.ndarray:
    """exp(-i beta (X_i X_j + Y_i Y_j)) as a full 2^n unitary.

    Acts as identity on aligned bit pairs and as a cos/sin block on the
    swap-coupled pair, so it preserves Hamming weight exactly.
    """
    dim = 2**n
    U = np.eye(dim, dtype=complex)
    c, s = np.cos(2 * beta), np.sin(2 * beta)
    for st in range(dim):
        if ((st >> i) & 1) == ((st >> j) & 1):
            continue
        sw = st ^ (1 << i) ^ (1 << j)
        if st < sw:
            U[st, st] = c
            U[sw, sw] = c
            U[st, sw] = -1j * s
            U[sw, st] = -1j * s
    return U

# ----------------------------------------------------------- the ansatz
class Brickwork:
    """U(1)-equivariant brickwork ansatz on the cycle C_n.

    Layer ell applies trainable Rz(theta[ell, q]) on every site, then fixed
    XY hopping exp(-i beta[ell, j] (XX + YY)) on the alternating edge set
    E_ell (even edges for even ell, odd edges for odd ell, cycle-closing
    edge included). Noise, when present, acts once after each layer.
    """
    INIT_SPREAD_SEED = 202   # fixed seed of the delocalised sector input

    def __init__(self, n: int, L: int, r: int = 1, init_state: str = "localized",
                 spread_seed: int = None):
        if n < 2:
            raise ValueError("need n >= 2")
        if init_state not in ("localized", "spread"):
            raise ValueError("init_state must be 'localized' or 'spread'")
        self.n, self.L, self.r = n, L, r
        self.init_state_mode = init_state
        self.spread_seed = int(spread_seed) if spread_seed is not None \
            else self.INIT_SPREAD_SEED
        self.dim = 2**n
        self.E = [[j for j in range(n) if j % 2 == ell % 2] for ell in range(L)]
        self.P_r = sector_projector(n, r)
        self.readout = self.P_r @ embed_one_qubit(PAULI_Z, 0, n) @ self.P_r

    # -- unitaries -----------------------------------------------------
    def z_layer(self, theta_ell: np.ndarray) -> np.ndarray:
        d = np.ones(self.dim, dtype=complex)
        for q in range(self.n):
            ph_m = np.exp(-1j * theta_ell[q] / 2)
            ph_p = np.exp(+1j * theta_ell[q] / 2)
            for st in range(self.dim):
                d[st] *= ph_m if ((st >> q) & 1) == 0 else ph_p
        return np.diag(d)

    def xy_layer(self, beta_ell: np.ndarray, ell: int) -> np.ndarray:
        U = np.eye(self.dim, dtype=complex)
        for j in self.E[ell]:
            U = xy_unitary(beta_ell[j], j, (j + 1) % self.n, self.n) @ U
        return U

    def layer_unitary(self, theta: np.ndarray, beta: np.ndarray, ell: int) -> np.ndarray:
        return self.xy_layer(beta[ell], ell) @ self.z_layer(theta[ell])

    # -- states --------------------------------------------------------
    def initial_state(self) -> np.ndarray:
        """Sector-r input state.

        "localized": the basis state |1^r 0^{n-r}> (excitations on sites
        0..r-1).  NOTE: at r = 2 with the primary shallow geometry (L = 3)
        this input carries no trainable-phase response for the Z_0 readout
        to numerical precision (the activity preflight returns a map at the
        numerical floor).  The inactivity is geometry-specific -- residual
        activity of order 1e-4 reappears at greater depth -- so this is a
        guarded configuration, not a universal r >= 2 statement.
        "spread": a fixed, seeded delocalised superposition over the full
        sector basis (seed INIT_SPREAD_SEED), which restores generic
        theta-response; higher-sector validation uses this input to avoid
        the inactive configuration.
        """
        psi = np.zeros(self.dim, dtype=complex)
        if self.init_state_mode == "localized":
            psi[(1 << self.r) - 1] = 1.0
            return psi
        sec = sector_basis(self.n, self.r)
        rng = np.random.default_rng(self.spread_seed)
        amps = rng.normal(size=len(sec)) + 1j * rng.normal(size=len(sec))
        amps = amps / np.linalg.norm(amps)
        for a, s in zip(amps, sec):
            psi[s] = a
        return psi

    def evolve_dm(self, theta: np.ndarray, beta: np.ndarray,
                  channel=None, gamma: float = 0.0) -> np.ndarray:
        """rho after L layers with per-layer noise (noiseless when gamma=0)."""
        psi = self.initial_state()
        rho = np.outer(psi, psi.conj())
        for ell in range(self.L):
            U = self.layer_unitary(theta, beta, ell)
            rho = U @ rho @ U.conj().T
            if channel is not None and gamma > 0:
                rho = channel.apply(rho, gamma, self.n)
        return rho

    def output(self, rho: np.ndarray) -> float:
        return float(np.real(np.trace(self.readout @ rho)))

def teacher_background(seed: int, L: int, n: int, scale: float = 0.5) -> np.ndarray:
    """Fixed random hopping background beta (part of the architecture)."""
    rng = np.random.default_rng(seed)
    return rng.uniform(-scale, scale, size=(L, n))

# ------------------------------------------------------------- channels
def apply_single_qubit_kraus(M: np.ndarray, Ks: list, q: int, n: int) -> np.ndarray:
    """Public helper. Applies a single-qubit Kraus map on site q of an
    n-qubit operator M and returns the result."""
    return _apply_1q_kraus(M, Ks, q, n)

def _apply_1q_kraus(M: np.ndarray, Ks: list, q: int, n: int) -> np.ndarray:
    """sum_K K_q M K_q^dag on an arbitrary 2^n x 2^n matrix via tensor reshape."""
    t = M.reshape((2,) * (2 * n))
    aL, aR = n - 1 - q, 2 * n - 1 - q
    out = np.zeros_like(t)
    for K in Ks:
        tmp = np.tensordot(K, t, axes=([1], [aL]))
        tmp = np.moveaxis(tmp, 0, aL)
        tmp = np.tensordot(K.conj(), tmp, axes=([1], [aR]))
        tmp = np.moveaxis(tmp, 0, aR)
        out = out + tmp
    return out.reshape(2**n, 2**n)

class NoiseChannel:
    """Base class. Subclasses either provide kraus_single(gamma) for uniform
    per-qubit action, or override apply() for structured action."""
    name = "base"
    def kraus_single(self, gamma: float) -> list:
        raise NotImplementedError
    def apply(self, M: np.ndarray, gamma: float, n: int) -> np.ndarray:
        Ks = self.kraus_single(gamma)
        for q in range(n):
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class AmplitudeDamping(NoiseChannel):
    name = "amp_damp"
    def kraus_single(self, g):
        return [np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=complex),
                np.array([[0, np.sqrt(g)], [0, 0]], dtype=complex)]

class Dephasing(NoiseChannel):
    name = "dephase"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g) * PAULI_Z]

class Depolarising(NoiseChannel):
    name = "depol"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g / 3) * PAULI_X,
                np.sqrt(g / 3) * PAULI_Y, np.sqrt(g / 3) * PAULI_Z]

class XError(NoiseChannel):
    """Bit-flip channel; breaks U(1). Retained as the symmetry-breaking probe."""
    name = "x_err"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g) * PAULI_X]

class InhomogeneousDephasing(NoiseChannel):
    """Per-qubit dephasing rates gamma * w_q, profile normalised to mean 1."""
    name = "inhom_dephase"
    def __init__(self, weights):
        w = np.asarray(weights, dtype=float)
        self.weights = w / np.mean(w)
    def apply(self, M, gamma, n):
        for q in range(n):
            gq = float(gamma * self.weights[q])
            if gq <= 0:
                continue
            Ks = [np.sqrt(1 - gq) * I2, np.sqrt(gq) * PAULI_Z]
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class SiteDependentAmpDamp(NoiseChannel):
    """Per-qubit amplitude-damping rates gamma * w_q, mean-normalised."""
    name = "site_amp_damp"
    def __init__(self, weights):
        w = np.asarray(weights, dtype=float)
        self.weights = w / np.mean(w)
    def apply(self, M, gamma, n):
        for q in range(n):
            gq = float(gamma * self.weights[q])
            if gq <= 0:
                continue
            Ks = [np.array([[1, 0], [0, np.sqrt(1 - gq)]], dtype=complex),
                  np.array([[0, np.sqrt(gq)], [0, 0]], dtype=complex)]
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class BiasedPauli(NoiseChannel):
    """Pauli noise with error ratios (r_X, r_Y, r_Z) summing to one."""
    # Convention. The probability tuple is ordered (p_X, p_Y, p_Z) and the
    # shipped benchmark uses (0.6, 0.2, 0.2), an X-dominated channel with
    # p_X != p_Y, which deliberately breaks phase covariance.
    name = "biased_pauli"
    def __init__(self, ratios=(0.6, 0.2, 0.2)):
        s = float(sum(ratios))
        self.ratios = tuple(x / s for x in ratios)
    def kraus_single(self, g):
        rX, rY, rZ = self.ratios
        return [np.sqrt(max(1 - g, 0.0)) * I2, np.sqrt(g * rX) * PAULI_X,
                np.sqrt(g * rY) * PAULI_Y, np.sqrt(g * rZ) * PAULI_Z]

class CoherentDissipativeMix(NoiseChannel):
    """Null coherent-component control. A uniform global Rz over-rotation
    (eps = eps_ratio * gamma, applied before amplitude damping at gamma)
    acts as a global phase within any fixed charge sector, so the coherent
    component is operationally invisible in-sector by construction and the
    channel must audit identically to amplitude damping. A genuinely active
    coherent perturbation is provided by SiteZOverRotation below."""
    name = "coh_diss_mix"
    def __init__(self, epsilon_ratio=0.5):
        self.epsilon_ratio = float(epsilon_ratio)
    def apply(self, M, gamma, n):
        eps = self.epsilon_ratio * gamma
        Urot = np.array([[np.exp(-1j * eps / 2), 0],
                         [0, np.exp(+1j * eps / 2)]], dtype=complex)
        for q in range(n):
            M = _apply_1q_kraus(M, [Urot], q, n)
        Ks = [np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex),
              np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)]
        for q in range(n):
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class CorrelatedDephasing(NoiseChannel):
    """Two-site ZZ dephasing on a fixed edge set:
        M -> (1 - gamma) M + gamma (Z_i Z_j) M (Z_i Z_j)   per edge.

    On the disjoint even-edge pairing {(2k, 2k+1)} in the single-excitation
    sector, coherences between sites of the SAME edge are exactly protected
    while cross-edge coherences decay -- the alignment-structured control.
    """
    name = "corr_dephase"
    def __init__(self, n, edges=None):
        if edges is None:
            edges = [(2 * k, 2 * k + 1) for k in range(n // 2)]
        self.edges = list(edges)
    def apply(self, M, gamma, n):
        for (i, j) in self.edges:
            t = M.reshape((2,) * (2 * n))
            tmp = t
            for q in (i, j):
                aL, aR = n - 1 - q, 2 * n - 1 - q
                tmp = np.tensordot(PAULI_Z, tmp, axes=([1], [aL]))
                tmp = np.moveaxis(tmp, 0, aL)
                tmp = np.tensordot(PAULI_Z.conj(), tmp, axes=([1], [aR]))
                tmp = np.moveaxis(tmp, 0, aR)
            M = ((1 - gamma) * t + gamma * tmp).reshape(2**n, 2**n)
        return M

class SiteZOverRotation(NoiseChannel):
    """Pure coherent site-dependent Z over-rotation (no dissipation).

    Applies exp(-i * gamma * c_q * Z_q / 2) on every site with the fixed
    non-uniform profile c_q = q / (n - 1). Because the profile is not
    proportional to the conserved total charge, the rotation acts
    non-trivially on in-sector coherences. Its restricted generator is
    anti-Hermitian in the Frobenius geometry, so lambda_coh vanishes and
    the audit reports a *signed* response susceptibility without a
    normalised alignment interpretation."""
    name = "site_z_overrotation"
    def apply(self, M, gamma, n):
        for q in range(n):
            c = q / (n - 1) if n > 1 else 0.0
            eps = gamma * c
            Urot = np.array([[np.exp(-1j * eps / 2), 0],
                             [0, np.exp(+1j * eps / 2)]], dtype=complex)
            M = _apply_1q_kraus(M, [Urot], q, n)
        return M

def build_channel_suite(n: int) -> dict:
    """The nine reference channels at system size n, isotropic first."""
    w = np.linspace(0.5, 1.5, n)
    return {
        "amp_damp":      AmplitudeDamping(),
        "dephase":       Dephasing(),
        "depol":         Depolarising(),
        "x_err":         XError(),
        "inhom_dephase": InhomogeneousDephasing(w),
        "site_amp_damp": SiteDependentAmpDamp(w),
        "biased_pauli":  BiasedPauli((0.6, 0.2, 0.2)),
        "coh_diss_mix":  CoherentDissipativeMix(0.5),
        "corr_dephase":  CorrelatedDephasing(n),
    }

ISOTROPIC = ("amp_damp", "dephase", "depol", "x_err")
STRUCTURED = ("inhom_dephase", "site_amp_damp", "biased_pauli",
              "coh_diss_mix", "corr_dephase")

def check_trace_preserving(channel, n: int, gamma: float = 0.05, seed: int = 0,
                           tol: float = 1e-10) -> float:
    """Max |Tr Phi(rho) - 1| over a few random density matrices."""
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(3):
        A = rng.normal(size=(2**n, 2**n)) + 1j * rng.normal(size=(2**n, 2**n))
        rho = A @ A.conj().T
        rho = rho / np.trace(rho)
        worst = max(worst, abs(float(np.real(np.trace(channel.apply(rho, gamma, n)))) - 1.0))
    return worst
'''
(PKG_DIR / "cohalign_core.py").write_text(COHALIGN_CORE_SRC)
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
import importlib
import cohalign_core as cac
importlib.reload(cac)
print(f"cohalign_core written and imported ({len(COHALIGN_CORE_SRC.splitlines())} lines)")

## Module 2 — `cohalign_rates`

Supporting library: the restricted generator, $\lambda_{\mathrm{coh}}$,
slot modes, the mode-rate diagnostic and the response-susceptibility
estimator with its bootstrap.


In [ ]:
COHALIGN_RATES_SRC = r'''"""
cohalign_rates.py -- generator-level coherence-rate estimators (the CohAlign
contribution).

Implements three objects on the in-sector off-diagonal block:

1.  lambda_coh  (worst-case sector coherence rate)
        The largest eigenvalue of the Hermitian part of -L_r, where
        L_r = (S - I)/gamma_probe and S is the channel superoperator
        restricted to the off-diagonal pair basis.  This is the
        variational worst case  sup_A -Re<A, L_r A> / <A, A>  and is
        computed here for EVERY channel, structured channels included
        (no analytic labels).

2.  lambda_vis_op  (operator-level aligned rate; quadratic estimator)
        The ensemble-averaged Rayleigh quotient of -L_r on the gradient
        mode  G_i(theta) = Pi_off,r [Z_qi, O_B(ins; theta)], where
        O_B(ins) is the readout Heisenberg-evolved back to the Rz
        insertion point (through all later layers AND the same layer's
        XY block), and the slot-ell mode is the forward conjugation of
        G_i to noise slot ell.  State-independent; this is the literal
        computable form of the aligned-rate Rayleigh quotient.

3.  response_rates  (response-weighted, layer-resolved aligned rates;
        the sharp a-priori predictor)
        Exact first-order degradation rates of the readout derivative,
        one per noise slot:
          post-insertion slots:  r_ell = -Re Tr(O_B^(ell) L_ch(D^(ell))) / df0
          pre-insertion  slots:  r_ell = -Re Tr(F^(ell)  L_ch(rho_ell)) / df0
        where D^(ell) is the derivative-carrying operator propagated
        forward from the insertion, F^(ell) is the adjoint response
        functional propagated backward, rho_ell is the noiseless state
        at slot ell, and df0 = Tr(O_B^(ell) D^(ell)) is the noiseless
        derivative (slot-invariant; used as an internal frame check).
        First-order prediction:  Delta_tilde ~= 2 * gamma * sum_ell r_ell.

The predicted alignment ratio reported by the audit is
    omega_pred = sum_ell r_ell / (L * lambda_coh),
directly comparable to the empirical  omega_hat = Delta_tilde / (2 gamma L
lambda_coh)  extracted from paired degradation measurements.
"""
import time
import numpy as np
from cohalign_core import (pair_basis, embed_one_qubit, PAULI_Z)

# ------------------------------------------------- restricted superoperator
def restricted_offdiag_superoperator(channel, n: int, r: int,
                                     gamma_probe: float = 1e-3) -> tuple:
    """Full matrix S of the channel on the in-sector off-diagonal pair basis.

    S[(c,d),(a,b)] = (Phi_gamma(|a><b|))[c,d]; off-diagonal-in-pair-basis
    leakage is captured exactly.  Returns (S, pairs).
    """
    pairs = pair_basis(n, r)
    K = len(pairs)
    dim = 2**n
    S = np.zeros((K, K), dtype=complex)
    for col, (a, b) in enumerate(pairs):
        E = np.zeros((dim, dim), dtype=complex)
        E[a, b] = 1.0
        Eo = channel.apply(E, gamma_probe, n)
        for row, (c, d) in enumerate(pairs):
            S[row, col] = Eo[c, d]
    return S, pairs

def restricted_generator(channel, n: int, r: int,
                         gamma_probe: float = 1e-3) -> tuple:
    """L_r = (S - I)/gamma_probe on the pair basis. Returns (L_r, pairs)."""
    S, pairs = restricted_offdiag_superoperator(channel, n, r, gamma_probe)
    return (S - np.eye(len(pairs))) / gamma_probe, pairs

def lambda_coh_worst(L_r: np.ndarray) -> float:
    """sup_{A != 0} -Re<A, L_r A>/<A, A> = max eig of the Hermitian part of -L_r."""
    H = -0.5 * (L_r + L_r.conj().T)
    return float(np.linalg.eigvalsh(H)[-1])

def lambda_coh_typical(L_r: np.ndarray) -> float:
    """Mean of the Hermitian-part spectrum (average contraction rate)."""
    H = -0.5 * (L_r + L_r.conj().T)
    return float(np.mean(np.linalg.eigvalsh(H)))

def rayleigh_rate(L_r: np.ndarray, g: np.ndarray) -> float:
    """-Re<g, L_r g> / <g, g> for a pair-basis vector g."""
    den = float(np.real(np.vdot(g, g)))
    if den < 1e-28:
        return float("nan")
    return float(-np.real(np.vdot(g, L_r @ g)) / den)

def vec_offdiag(M: np.ndarray, pairs: list) -> np.ndarray:
    return np.array([M[a, b] for (a, b) in pairs], dtype=complex)

# --------------------------------------------------- gradient-mode geometry
def insertion_readout(model, theta: np.ndarray, beta: np.ndarray,
                      ell_i: int) -> np.ndarray:
    """Heisenberg readout at the Rz insertion point of layer ell_i:
    evolved back through all layers > ell_i and the XY block of layer ell_i."""
    U_back = model.xy_layer(beta[ell_i], ell_i)
    for ell in range(ell_i + 1, model.L):
        U_back = model.layer_unitary(theta, beta, ell) @ U_back
    return U_back.conj().T @ model.readout @ U_back

def slot_modes(model, theta: np.ndarray, beta: np.ndarray,
               ell_i: int, q_i: int) -> dict:
    """Gradient mode G = [Z_qi, O_B(ins)] forward-conjugated to each noise
    slot ell in [ell_i, L-1]. Returns {ell: mode matrix}."""
    Zq = embed_one_qubit(PAULI_Z, q_i, model.n)
    OB_ins = insertion_readout(model, theta, beta, ell_i)
    G = Zq @ OB_ins - OB_ins @ Zq
    W = model.xy_layer(beta[ell_i], ell_i)
    M = W @ G @ W.conj().T
    modes = {ell_i: M}
    for ell in range(ell_i + 1, model.L):
        Ul = model.layer_unitary(theta, beta, ell)
        M = Ul @ M @ Ul.conj().T
        modes[ell] = M
    return modes

# ------------------------------------- estimator 1: quadratic (Eq.-7 literal)
def lambda_mode(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                  n_theta: int = 20, delta_init: float = 0.05,
                  seed: int = 1, gamma_probe: float = 1e-3,
                  L_r=None, pairs=None, theta_draws=None) -> dict:
    """Operator mode contraction diagnostic. Ensemble-averaged Rayleigh
    quotient of -L_r on the slot modes of the derivative carrying operator.
    This is a state-independent operator diagnostic, not a visibility
    measure, and it does not by itself predict the measured response.

    Returns {"lambda_vis_op", "per_slot", "lambda_coh"}; per_slot maps each
    noise slot ell >= ell_i to its mean quadratic rate.
    """
    if L_r is None or pairs is None:
        L_r, pairs = restricted_generator(channel, model.n, model.r,
                                          gamma_probe)
    lam_coh = lambda_coh_worst(L_r)
    rng = np.random.default_rng(seed)
    per = {ell: [] for ell in range(ell_i, model.L)}
    if theta_draws is not None:
        n_theta = len(theta_draws)
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init,
                                  size=(model.L, model.n)))
        for ell, M in slot_modes(model, theta, beta, ell_i, q_i).items():
            g = vec_offdiag(M, pairs)
            v = rayleigh_rate(L_r, g)
            if np.isfinite(v):
                per[ell].append(v)
    per_slot = {ell: (float(np.mean(v)) if v else float("nan"))
                for ell, v in per.items()}
    vals = [v for v in per_slot.values() if np.isfinite(v)]
    return {"lambda_vis_op": float(np.mean(vals)) if vals else float("nan"),
            "per_slot": per_slot, "lambda_coh": lam_coh}

# ------------------- estimator 2: response-weighted, layer-resolved (sharp)
def lambda_vis_op(*args, **kwargs):
    """Deprecated alias for lambda_mode, retained for backward
    compatibility with earlier releases."""
    import warnings
    warnings.warn("lambda_vis_op is deprecated; use lambda_mode",
                  DeprecationWarning, stacklevel=2)
    return lambda_mode(*args, **kwargs)

def response_rates(model, theta: np.ndarray, beta: np.ndarray, channel,
                   ell_i: int, q_i: int, gamma_probe: float = 1e-3,
                   frame_check: bool = False) -> dict:
    """Exact first-order aligned rates, one per noise slot (all L slots).

    Returns {"rates": {ell: r_ell}, "df0": noiseless derivative,
             "frame_defect": worst frame-invariance violation if checked}.
    Delta_tilde_pred = 2 * gamma * sum(rates.values()).
    """
    n, L = model.n, model.L
    Zq = embed_one_qubit(PAULI_Z, q_i, n)
    # noiseless states after each layer; state at the insertion point
    psi = model.initial_state()
    rho = np.outer(psi, psi.conj())
    rho_slot = []
    rho_pre_ins = None
    for ell in range(L):
        Uz = model.z_layer(theta[ell])
        Ux = model.xy_layer(beta[ell], ell)
        rho_z = Uz @ rho @ Uz.conj().T
        if ell == ell_i:
            rho_pre_ins = rho_z
        rho = Ux @ rho_z @ Ux.conj().T
        rho_slot.append(rho)
    # derivative-carrying operator, forward from the insertion
    C = -0.5j * (Zq @ rho_pre_ins - rho_pre_ins @ Zq)
    Ux_i = model.xy_layer(beta[ell_i], ell_i)
    D = Ux_i @ C @ Ux_i.conj().T
    D_slot = {ell_i: D}
    for ell in range(ell_i + 1, L):
        Ul = model.layer_unitary(theta, beta, ell)
        D = Ul @ D @ Ul.conj().T
        D_slot[ell] = D
    # Heisenberg readout at each slot
    OB_slot = {L - 1: model.readout}
    for ell in range(L - 2, -1, -1):
        Ul = model.layer_unitary(theta, beta, ell + 1)
        OB_slot[ell] = Ul.conj().T @ OB_slot[ell + 1] @ Ul
    df0 = float(np.real(np.trace(OB_slot[ell_i] @ D_slot[ell_i])))
    # adjoint response functional for pre-insertion slots
    OB_ins = Ux_i.conj().T @ OB_slot[ell_i] @ Ux_i
    F = -0.5j * (OB_ins @ Zq - Zq @ OB_ins)
    Uz_i = model.z_layer(theta[ell_i])
    F = Uz_i.conj().T @ F @ Uz_i
    F_slot = {}
    if ell_i >= 1:
        F_slot[ell_i - 1] = F
        for ell in range(ell_i - 2, -1, -1):
            Ul = model.layer_unitary(theta, beta, ell + 1)
            F_slot[ell] = Ul.conj().T @ F_slot[ell + 1] @ Ul
    frame_defect = 0.0
    if frame_check and abs(df0) > 1e-16:
        for ell in range(ell_i, L):
            v = float(np.real(np.trace(OB_slot[ell] @ D_slot[ell])))
            frame_defect = max(frame_defect, abs(v - df0) / abs(df0))
        for ell in F_slot:
            v = float(np.real(np.trace(F_slot[ell] @ rho_slot[ell])))
            frame_defect = max(frame_defect, abs(v - df0) / abs(df0))
    Lch = lambda X: (channel.apply(X, gamma_probe, n) - X) / gamma_probe
    # raw first-order response terms h_ell (no division by the derivative)
    h_slots = {}
    for ell in range(ell_i, L):
        h_slots[ell] = float(np.real(np.trace(OB_slot[ell] @ Lch(D_slot[ell]))))
    for ell in F_slot:
        h_slots[ell] = float(np.real(np.trace(F_slot[ell] @ Lch(rho_slot[ell]))))
    # per-draw ratios r_ell = -h_ell / g0 as an OPTIONAL diagnostic only
    if abs(df0) > 1e-14:
        rates = {ell: float(-h / df0) for ell, h in h_slots.items()}
    else:
        rates = {ell: float("nan") for ell in h_slots}
    return {"rates": rates, "h_slots": h_slots, "df0": df0,
            "frame_defect": frame_defect}

def response_terms(model, theta, beta, channel, ell_i, q_i,
                   gamma_probe=1e-3, frame_check=False):
    """Raw first-order response terms for one draw. Returns
    {"g0": noiseless derivative, "h_slots": {ell: h_ell}, "frame_defect"}.
    The cross-moment estimator aggregates these without ever dividing by
    an individual draw."""
    out = response_rates(model, theta, beta, channel, ell_i, q_i,
                         gamma_probe, frame_check)
    return {"g0": out["df0"], "h_slots": out["h_slots"],
            "frame_defect": out["frame_defect"]}

def lambda_vis_resp(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                    n_theta: int = 20, delta_init: float = 0.05,
                    seed: int = 1, gamma_probe: float = 1e-3,
                    bootstrap_B: int = 200, bootstrap_seed=None,
                    theta_draws=None) -> dict:
    """Ensemble response-weighted rates over the small-box prior.

    Aggregation is the derivative-squared-weighted mean,
        rate_ell = sum_draws df0^2 r_ell / sum_draws df0^2,
    which is the exact first-order object matched by the paired M2 ratio:
        M2(gamma)/M2(0) = <df0^2 (1 - 2 gamma sum_ell r_ell)> / <df0^2>
                        = 1 - 2 gamma sum_ell rate_ell + O(gamma^2).
    A plain mean of per-draw rate ratios is a mean-of-ratios and becomes
    unstable whenever df0 varies strongly across the prior (deep circuits);
    the weighted form is identical where df0 is stable and remains finite
    everywhere.  The effective sample size ESS = (sum w)^2 / sum w^2 and a
    weighted-bootstrap 95% CI on rate_sum quantify ensemble uncertainty.

    Returns {"rate_sum", "rate_sum_ci": (lo, hi), "per_slot", "n_used",
             "ess"}.
    """
    if theta_draws is not None:
        n_theta = len(theta_draws)
    rng = np.random.default_rng(seed)
    # Direct cross-moment accumulation in a SINGLE pass over the draws.
    # If theta_draws is supplied it is used verbatim, which lets a matched
    # finite-difference reference share EXACTLY the same parameter draws.
    # For each slot, N_ell = sum_j (-g0_j * h_ell_j) and D = sum_j g0_j^2,
    # with Lambda_ell = N_ell / D. No per-draw division is performed, every
    # sampled draw enters, and per-draw numerator and denominator arrays are
    # retained so the bootstrap is a direct ratio-of-sums resample with no
    # second execution of the response calculation.
    Lp = model.L
    num_draw = np.zeros((n_theta, Lp))
    den_draw = np.zeros(n_theta)
    k = 0
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init,
                                  size=(Lp, model.n)))
        t = response_terms(model, theta, beta, channel, ell_i, q_i,
                           gamma_probe)
        g0 = t["g0"]
        if not np.isfinite(g0):
            continue
        for ell, h in t["h_slots"].items():
            num_draw[k, ell] = -g0 * h
        den_draw[k] = g0 * g0
        k += 1
    num_draw, den_draw = num_draw[:k], den_draw[:k]
    if k == 0 or den_draw.sum() <= 0.0:
        return {"rate_sum": float("nan"), "rate_sum_ci": (float("nan"),) * 2,
                "per_slot": {ell: float("nan") for ell in range(Lp)},
                "n_used": 0, "ess": 0.0,
                "bootstrap_B": int(bootstrap_B),
                "bootstrap_seed": int(bootstrap_seed
                                      if bootstrap_seed is not None
                                      else seed + 10_000),
                "num_draw": num_draw, "den_draw": den_draw}
    D = float(den_draw.sum())
    per_slot = {ell: float(num_draw[:, ell].sum() / D) for ell in range(Lp)}
    rate_sum = float(num_draw.sum() / D)
    ess = float(D ** 2 / np.sum(den_draw ** 2))  # weight concentration
    if bootstrap_seed is None:
        bootstrap_seed = seed + 10_000
    brng = np.random.default_rng(bootstrap_seed)
    tot = num_draw.sum(axis=1)
    stats = []
    for _ in range(bootstrap_B):
        idx = brng.integers(0, k, size=k)
        dd = den_draw[idx].sum()
        if dd > 0:
            stats.append(float(tot[idx].sum() / dd))
    lo, hi = (np.percentile(stats, [2.5, 97.5]) if stats
              else (float("nan"), float("nan")))
    return {"rate_sum": rate_sum, "rate_sum_ci": (float(lo), float(hi)),
            "per_slot": per_slot, "n_used": int(k), "ess": ess,
            "bootstrap_B": int(bootstrap_B),
            "bootstrap_seed": int(bootstrap_seed),
            "num_draw": num_draw, "den_draw": den_draw}

def prepare_channel_context(channel, n, r, gamma_probe=1e-3):
    """Build the restricted generator ONCE and derive everything that does
    not depend on the parameter. Returns a dict context consumed by
    audit_parameter, so auditing many parameters of one channel reuses the
    expensive construction. The normalisation status is classified from the
    relative Frobenius norms of the Hermitian and anti-Hermitian parts of
    the restricted generator rather than from an absolute rate threshold."""
    L_r, pairs = restricted_generator(channel, n, r, gamma_probe)
    H = -0.5 * (L_r + L_r.conj().T)
    A = 0.5 * (L_r - L_r.conj().T)
    nH = float(np.linalg.norm(H)); nA = float(np.linalg.norm(A))
    if nH >= 10.0 * nA or nA == 0.0:
        status = "contractive"
    elif nA >= 10.0 * nH:
        status = "coherent_dominated"
    else:
        status = "mixed"
    return {"channel": channel, "n": n, "r": r,
            "gamma_probe": gamma_probe, "L_r": L_r, "pairs": pairs,
            "lambda_coh_worst": lambda_coh_worst(L_r),
            "lambda_coh_typical": lambda_coh_typical(L_r),
            "herm_norm": nH, "antiherm_norm": nA,
            "normalisation_status": status}

def lambda_mode_from_context(ctx, model, beta, ell_i, q_i, n_theta=20,
                             delta_init=0.05, seed=1, theta_draws=None):
    """Operator mode contraction diagnostic computed from a prepared channel
    context; the restricted generator is never rebuilt here."""
    return lambda_mode(model, beta, ctx["channel"], ell_i, q_i,
                         n_theta=n_theta, delta_init=delta_init, seed=seed,
                         gamma_probe=ctx["gamma_probe"],
                         L_r=ctx["L_r"], pairs=ctx["pairs"],
                         theta_draws=theta_draws)

def audit_parameter(ctx, model, beta, ell_i, q_i, n_theta=20,
                    delta_init=0.05, seed=1, theta_draws=None,
                    bootstrap_B=200, bootstrap_seed=None):
    """Audit one parameter using a prepared channel context. The generator
    is not rebuilt. omega values are reported only when the context status
    is contractive; otherwise the signed susceptibility is the output."""
    t0 = time.time()
    channel = ctx["channel"]; gamma_probe = ctx["gamma_probe"]
    lam_w = ctx["lambda_coh_worst"]; lam_t = ctx["lambda_coh_typical"]
    op = lambda_mode_from_context(ctx, model, beta, ell_i, q_i,
                                  n_theta=n_theta, delta_init=delta_init,
                                  seed=seed, theta_draws=theta_draws)
    rp = lambda_vis_resp(model, beta, channel, ell_i, q_i,
                         n_theta=n_theta, delta_init=delta_init,
                         seed=seed, gamma_probe=gamma_probe,
                         bootstrap_B=bootstrap_B,
                         bootstrap_seed=bootstrap_seed,
                         theta_draws=theta_draws)
    contractive = ctx["normalisation_status"] == "contractive" and lam_w > 0
    omega_op = op["lambda_vis_op"] / lam_w if contractive else float("nan")
    omega_resp = (rp["rate_sum"] / (model.L * lam_w)
                  if contractive else float("nan"))
    ci = rp["rate_sum_ci"]
    omega_resp_ci = (tuple(c / (model.L * lam_w) for c in ci)
                     if contractive else (float("nan"),) * 2)
    out = {"channel": getattr(channel, "name", type(channel).__name__),
           "n": model.n, "L": model.L, "r": model.r,
           "ell_i": ell_i, "q_i": q_i,
           "lambda_coh_worst": lam_w, "lambda_coh_typical": lam_t,
           "normalisation_status": ctx["normalisation_status"],
           "lambda_mode": op["lambda_vis_op"], "omega_mode": omega_op,
           "lambda_response": rp["rate_sum"],
           "lambda_response_ci": rp["rate_sum_ci"],
           "omega_response": omega_resp, "omega_response_ci": omega_resp_ci,
           "per_slot_response": rp["per_slot"],
           "signed_susceptibility_only": not contractive,
           "ess": rp["ess"], "n_theta_used": rp["n_used"],
           "bootstrap_B": rp["bootstrap_B"],
           "bootstrap_seed": rp["bootstrap_seed"],
           "wall_seconds": time.time() - t0,
           # deprecated aliases retained for backward compatibility
           "lambda_vis_op": op["lambda_vis_op"], "omega_op": omega_op,
           "rate_sum_resp": rp["rate_sum"], "omega_resp": omega_resp,
           "omega_resp_ci": omega_resp_ci,
           "per_slot_resp": rp["per_slot"], "per_slot_op": op["per_slot"]}
    if not contractive:
        out["note"] = ("normalisation status is "
                       f"{ctx['normalisation_status']}; the audit reports "
                       "the signed response susceptibility and no "
                       "normalised alignment ratio")
    return out

def audit_parameters(ctx, model, beta, params, n_theta=20,
                     delta_init=0.05, seed=1):
    """Audit a list of (ell, q) parameters with one shared context."""
    return [audit_parameter(ctx, model, beta, e, q, n_theta=n_theta,
                            delta_init=delta_init, seed=seed)
            for (e, q) in params]

def audit_channel(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                  n_theta: int = 20, delta_init: float = 0.05,
                  seed: int = 1, gamma_probe: float = 1e-3) -> dict:
    """One-call audit of a (channel, architecture, parameter) triple.
    Equivalent to prepare_channel_context followed by audit_parameter; the
    restricted generator is built exactly once."""
    t0 = time.time()
    ctx = prepare_channel_context(channel, model.n, model.r, gamma_probe)
    out = audit_parameter(ctx, model, beta, ell_i, q_i, n_theta=n_theta,
                          delta_init=delta_init, seed=seed)
    out["wall_seconds"] = time.time() - t0
    return out


'''
(PKG_DIR / "cohalign_rates.py").write_text(COHALIGN_RATES_SRC)
import cohalign_rates as car
importlib.reload(car)
print(f"cohalign_rates written and imported ({len(COHALIGN_RATES_SRC.splitlines())} lines)")

## Module 3 — `cohalign_bench`

Supporting library: activity preflight, parameter selection, CRN paired
degradation and the empirical alignment ratio.


In [ ]:
COHALIGN_BENCH_SRC = r'''"""
cohalign_bench.py -- empirical validation machinery for CohAlign.

Provides:
  - activity_preflight / pick_parameters: locate readout-visible (active)
    parameters via the noiseless parameter-shift gradient map, and select
    validation parameters spanning the alignment range
  - paired_degradation: common-random-numbers paired measurement of the
    relative squared-gradient degradation Delta_tilde = 1 - M2(gamma)/M2(0)
    with bootstrap confidence intervals (shared theta draws between the
    noiseless and noisy passes, and across channels)
  - omega_hat: empirical alignment ratio Delta_tilde / (2 gamma L lambda_coh)
  - ols_loglog: ordinary least squares on log-log design matrices with R^2,
    RMSE and coefficient table (used for the predictor-comparison phase)
"""
import numpy as np

# ------------------------------------------------------ activity preflight
def parameter_shift_grad(model, theta, beta, ell, q, channel=None,
                         gamma=0.0, shift=np.pi / 2):
    tp, tm = theta.copy(), theta.copy()
    tp[ell, q] += shift
    tm[ell, q] -= shift
    fp = model.output(model.evolve_dm(tp, beta, channel, gamma))
    fm = model.output(model.evolve_dm(tm, beta, channel, gamma))
    return 0.5 * (fp - fm)

def activity_preflight(model, beta, n_draw=4, delta_init=0.05, seed=11):
    """Mean squared noiseless gradient for every (ell, q). O(n L) circuit pairs
    per draw; identifies the backward light cone of the readout."""
    act = np.zeros((model.L, model.n))
    rng = np.random.default_rng(seed)
    for _ in range(n_draw):
        theta = rng.uniform(-delta_init, delta_init, size=(model.L, model.n))
        for ell in range(model.L):
            for q in range(model.n):
                g = parameter_shift_grad(model, theta, beta, ell, q)
                act[ell, q] += g * g
    return act / n_draw

def pick_parameters(act, k=2, floor_frac=1e-4, abs_floor=1e-20,
                    distinct_layers=True):
    """Top-k active parameters by mean squared gradient (descending).

    With distinct_layers=True (default) the selected parameters are drawn
    from k different layers, so validation covers genuinely distinct (not
    symmetry-equivalent) parameter locations.

    Raises if the whole map is numerically zero -- the signature of an input
    state that carries no trainable-phase response for this readout at this
    geometry (observed for the localised basis-state input at r = 2, L = 3;
    the inactivity is geometry-specific, not universal for r >= 2)."""
    if act.max() < abs_floor:
        raise RuntimeError(
            "activity preflight found no readout-visible parameters "
            f"(max activity {act.max():.2e}); the input state carries no "
            "trainable-phase response for this readout at this geometry -- "
            "for higher sectors consider Brickwork(..., init_state='spread')")
    flat = [(-act[ell, q], ell, q)
            for ell in range(act.shape[0]) for q in range(act.shape[1])
            if act[ell, q] > floor_frac * act.max()]
    flat.sort()
    if not distinct_layers:
        return [(ell, q) for (_, ell, q) in flat[:k]]
    out, used_layers = [], set()
    for (_, ell, q) in flat:
        if ell in used_layers:
            continue
        out.append((ell, q)); used_layers.add(ell)
        if len(out) == k:
            break
    # fall back to plain top-k if fewer than k layers are active
    if len(out) < k:
        out = [(ell, q) for (_, ell, q) in flat[:k]]
    return out

# ------------------------------------------------- CRN paired degradation
def paired_degradation(model, beta, ell, q, channel, gamma,
                       n_theta=8, delta_init=0.05, seed=1234,
                       shift=np.pi / 2, bootstrap_B=200, boot_seed=7,
                       g0_floor=1e-12, return_draws=False,
                       theta_draws=None):
    """CRN paired estimate of Delta_tilde = 1 - M2(gamma)/M2(0).

    The SAME theta draws feed the noiseless and noisy passes (and, because
    the seed is caller-fixed, the same draws are shared across channels and
    gamma values).  M2 is the mean squared parameter-shift derivative over
    the retained draws; the bootstrap resamples draw indices.
    Returns dict with m2_zero, m2_noise, delta_tilde, ci_lo, ci_hi, n_used;
    with return_draws=True also g0_sq and gn_sq (per-draw squared gradients,
    aligned across gamma values by the shared CRN seed) for joint bootstraps.
    """
    rng = np.random.default_rng(seed)
    g0_sq, gn_sq = [], []
    if theta_draws is not None:
        n_theta = len(theta_draws)
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init, size=(model.L, model.n)))
        g0 = parameter_shift_grad(model, theta, beta, ell, q, None, 0.0, shift)
        if g0 * g0 < g0_floor:
            continue
        gn = parameter_shift_grad(model, theta, beta, ell, q, channel, gamma, shift)
        g0_sq.append(g0 * g0)
        gn_sq.append(gn * gn)
    g0_sq, gn_sq = np.asarray(g0_sq), np.asarray(gn_sq)
    if len(g0_sq) == 0:
        return {"m2_zero": np.nan, "m2_noise": np.nan, "delta_tilde": np.nan,
                "ci_lo": np.nan, "ci_hi": np.nan, "n_used": 0}
    m2_0, m2_g = float(np.mean(g0_sq)), float(np.mean(gn_sq))
    delta = 1.0 - m2_g / m2_0
    brng = np.random.default_rng(boot_seed)
    stats = []
    for _ in range(bootstrap_B):
        idx = brng.integers(0, len(g0_sq), size=len(g0_sq))
        stats.append(1.0 - np.mean(gn_sq[idx]) / np.mean(g0_sq[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    out = {"m2_zero": m2_0, "m2_noise": m2_g, "delta_tilde": float(delta),
           "ci_lo": float(lo), "ci_hi": float(hi), "n_used": int(len(g0_sq))}
    if return_draws:
        out["g0_sq"] = g0_sq
        out["gn_sq"] = gn_sq
    return out

def omega_hat(delta_tilde, gamma, L, lambda_coh):
    """Empirical alignment ratio Delta_tilde / (2 gamma L lambda_coh)."""
    den = 2.0 * gamma * L * lambda_coh
    return float(delta_tilde / den) if den > 0 else float("nan")

# --------------------------------------------------------------- regression
def ols_loglog(y, X_cols, names):
    """OLS of y on [1, X_cols...]. Returns dict with R2, RMSE, coefficients."""
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, dtype=float)
                                             for c in X_cols])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    yhat = X @ b
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return {"R2": 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
            "RMSE": float(np.sqrt(ss_res / len(y))),
            "coef": dict(zip(["intercept"] + list(names), [float(v) for v in b])),
            "n": int(len(y))}
'''
(PKG_DIR / "cohalign_bench.py").write_text(COHALIGN_BENCH_SRC)
import cohalign_bench as cab
importlib.reload(cab)
print(f"cohalign_bench written and imported ({len(COHALIGN_BENCH_SRC.splitlines())} lines)")

# shared architecture objects for the primary configuration
N, L, R = CFG["n"], CFG["L"], CFG["r"]
MODEL = cac.Brickwork(N, L, R)
BETA  = cac.teacher_background(TEACHER_SEED, L, N)
SUITE = cac.build_channel_suite(N)
print(f"primary architecture: n={N}, L={L}, r={R}; channels: {list(SUITE)}")

## Module 4 — `cgl_graph`

The coherence-graph layer: analytic pairwise rates, the difference and
potential form certificates, protection graph and kernel, the combinatorial
cut-crossing predictor, visible pair weights, the support-cut bound,
slot-resolved noise, and spectral helpers.


In [ ]:
CGL_GRAPH_SRC = r'''"""
cgl_graph.py -- the coherence-graph layer (the new contribution of this
pipeline). Builds on cohalign_core / cohalign_rates, which are materialised
verbatim from the CohAlign v1.8.1 release so that every rate, mode and
response quantity used here is bit-identical to the published pipeline.

Objects implemented:

1.  Analytic pairwise decay rates (per unit gamma, first order) for every
    phase-covariant channel of the CohAlign suite, in any charge sector:
      dephasing family   rate(a,b) = 2 sum_q w_q [bit_q differs]
      correlated ZZ      rate(a,b) = 2 sum_e   [edge product differs]
      damping family     rate(a,b) = phi(a) + phi(b),
                         phi(a) = (1/2) sum_{q in exc(a)} w_q
      depolarising       rate(a,b) = (2/3)(n - d) + (4/3) d,  d = Hamming
      biased Pauli       rate(a,b) = n (rX + rY) + 2 rZ d
      X error            rate(a,b) = n            (leakage probe; in-sector)
    These are exact first-order values; the numeric finite-probe generator
    carries the documented O(gamma_probe) bias on top.

2.  The structural decomposition certificates.
      difference form  (graph-Laplacian / squared-distance):
        rate(a,b) = || v(a) - v(b) ||^2  with  v_k(a) = sqrt(c_k) l_k(a)
        over the jump spectra l_k, for the Z-diagonal (dephasing) family.
      potential form (Schroedinger):
        rate(a,b) = phi(a) + phi(b)  for the damping family.
    Both are verified to machine precision channel by channel.

3.  The protection graph and its kernel: pairs (a,b) with rate < tol form
    the zero-rate edge set; its ordered-pair count must equal the kernel
    dimension of the Hermitian part of -L_r, and its connected components
    organise the protected subspace.

4.  The combinatorial cut-crossing predictor: pure edge-set reachability
    (no linear algebra) computing, per noise slot, the visible interaction
    cone I(ell) and asking whether any unprotected 2-subset of I(ell) is
    present. "All slots protected" is a sufficient condition for zero
    response; its negation is the exposure prediction tested in Phase C.

5.  Visible pair weights: the pairwise decomposition of the first-order
    response pairing, giving the fraction of visible weight on unprotected
    coherence pairs (the quantitative companion of the cone predictor).

6.  The support-cut (Cheeger-type) lower bound on the mode rate for
    diagonal-generator channels, and helpers for the adversarial sweep.

7.  Slot-resolved noise (independent per-slot strengths) for the
    second-order / biharmonic phase, plus Fiedler-value helpers for the
    higher-sector spectral analysis.
"""
import numpy as np
from itertools import combinations
from cohalign_core import (sector_basis, pair_basis, embed_one_qubit,
                           PAULI_Z)

TOL_RATE_ZERO = 1e-9


# ------------------------------------------------------------- bit helpers
def _bits(a, n):
    return [(a >> q) & 1 for q in range(n)]

def _z(a, q):
    return -1.0 if ((a >> q) & 1) else 1.0

def _exc(a, n):
    return [q for q in range(n) if (a >> q) & 1]

def _hamming(a, b):
    return bin(a ^ b).count("1")


# --------------------------------------------- analytic first-order rates
def analytic_rate_pair(channel, a, b, n):
    """Exact first-order decay rate (per unit gamma) of the coherence
    |a><b| under the named CohAlign channel, restricted to the in-sector
    block. Real part only; the null coherent-dissipative control maps to
    its amplitude-damping dissipative part. Raises for channels without a
    phase-covariant diagonal form (none in the shipped suite except the
    pure coherent over-rotation, whose Hermitian rate is zero)."""
    name = channel.name
    if name in ("dephase", "inhom_dephase"):
        w = getattr(channel, "weights", np.ones(n))
        return 2.0 * sum(float(w[q]) for q in range(n)
                         if ((a >> q) & 1) != ((b >> q) & 1))
    if name == "corr_dephase":
        tot = 0.0
        for (i, j) in channel.edges:
            pa = _z(a, i) * _z(a, j)
            pb = _z(b, i) * _z(b, j)
            if pa * pb < 0:
                tot += 2.0
        return tot
    if name in ("amp_damp", "site_amp_damp", "coh_diss_mix"):
        w = getattr(channel, "weights", np.ones(n))
        return _damping_potential(a, n, w) + _damping_potential(b, n, w)
    if name == "depol":
        d = _hamming(a, b)
        return (2.0 / 3.0) * (n - d) + (4.0 / 3.0) * d
    if name == "biased_pauli":
        rX, rY, rZ = channel.ratios
        return n * (rX + rY) + 2.0 * rZ * _hamming(a, b)
    if name == "x_err":
        return float(n)
    if name == "site_z_overrotation":
        return 0.0
    raise ValueError(f"no analytic diagonal form for channel '{name}'")

def _damping_potential(a, n, w):
    return 0.5 * sum(float(w[q]) for q in _exc(a, n))

def analytic_rate_vector(channel, n, r):
    pairs = pair_basis(n, r)
    return np.array([analytic_rate_pair(channel, a, b, n)
                     for (a, b) in pairs]), pairs


# ------------------------------------------ structural form certificates
def difference_form_embedding(channel, n):
    """Jump-spectrum embedding v(a) for the Z-diagonal family, such that
    rate(a,b) = ||v(a) - v(b)||^2 exactly. Returns a callable a -> vector.
    """
    name = channel.name
    if name in ("dephase", "inhom_dephase"):
        w = getattr(channel, "weights", np.ones(n))
        def v(a):
            return np.array([np.sqrt(w[q] / 2.0) * _z(a, q)
                             for q in range(n)])
        return v
    if name == "corr_dephase":
        edges = channel.edges
        def v(a):
            return np.array([np.sqrt(0.5) * _z(a, i) * _z(a, j)
                             for (i, j) in edges])
        return v
    raise ValueError(f"'{name}' is not in the Z-diagonal difference family")

def check_difference_form(channel, n, r):
    """max_ab | rate_analytic(a,b) - ||v(a)-v(b)||^2 | over the sector.
    An algebraic identity; the return value should be at machine precision.
    """
    v = difference_form_embedding(channel, n)
    worst = 0.0
    for (a, b) in pair_basis(n, r):
        pred = analytic_rate_pair(channel, a, b, n)
        emb = float(np.sum((v(a) - v(b)) ** 2))
        worst = max(worst, abs(pred - emb))
    return worst

def check_potential_form(channel, n, r):
    """max_ab | rate_analytic(a,b) - (phi(a)+phi(b)) | for the damping
    family (Schroedinger / vertex-potential form)."""
    w = getattr(channel, "weights", np.ones(n))
    worst = 0.0
    for (a, b) in pair_basis(n, r):
        pred = analytic_rate_pair(channel, a, b, n)
        pot = _damping_potential(a, n, w) + _damping_potential(b, n, w)
        worst = max(worst, abs(pred - pot))
    return worst


# ----------------------------------------------- numeric generator probes
def offdiag_mass(L_r):
    """Relative off-pair-diagonal mass of the restricted generator
    (diagonality certificate for the phase-covariant diagonal families)."""
    A = np.abs(L_r)
    d = float(np.trace(A))
    tot = float(A.sum())
    off = tot - d
    return off / max(tot, 1e-30)

def numeric_diag_rates(L_r):
    return -np.real(np.diag(L_r))

def hermitian_rate_spectrum(L_r):
    H = -0.5 * (L_r + L_r.conj().T)
    return np.linalg.eigvalsh(H)


# ------------------------------------------------ protection graph, kernel
def protected_mask(rates, tol=TOL_RATE_ZERO):
    return np.asarray(rates) < tol

def protection_components(n, r, mask, pairs):
    """Connected components of the protection graph on sector basis states
    (edges = zero-rate coherence pairs). Returns (n_components, sizes)."""
    sec = sector_basis(n, r)
    idx = {s: k for k, s in enumerate(sec)}
    parent = list(range(len(sec)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    for m, (a, b) in zip(mask, pairs):
        if m:
            union(idx[a], idx[b])
    roots = {}
    for k in range(len(sec)):
        roots.setdefault(find(k), 0)
        roots[find(k)] += 1
    sizes = sorted(roots.values(), reverse=True)
    return len(sizes), sizes

def kernel_dimension(L_r, tol=1e-6):
    ev = hermitian_rate_spectrum(L_r)
    return int(np.sum(np.abs(ev) < tol))


# ---------------------------------------- combinatorial cut-crossing cones
def _grow(S, edge_sites, n):
    """One layer of support growth through the edge set
    {(j, j+1 mod n) : j in edge_sites} (same both directions)."""
    S = set(S)
    out = set(S)
    for j in edge_sites:
        k = (j + 1) % n
        if j in S:
            out.add(k)
        if k in S:
            out.add(j)
    return out

def cones(n, L, r, ell_i, q_i, edge_sets):
    """All cones needed by the cut predictor. edge_sets[ell] is the list of
    left sites of layer ell's hopping edges (Brickwork.E)."""
    state = {}
    S = set(range(r))                    # localised input, excitations 0..r-1
    for ell in range(L):
        S = _grow(S, edge_sets[ell], n)
        state[ell] = set(S)              # support after layer ell (slot ell)
    ob = {L - 1: {0}}
    for ell in range(L - 2, -1, -1):
        ob[ell] = _grow(ob[ell + 1], edge_sets[ell + 1], n)
    f = {}
    if ell_i >= 1:
        f_start = _grow(ob[ell_i], edge_sets[ell_i], n) | {q_i}
        f[ell_i - 1] = f_start
        for ell in range(ell_i - 2, -1, -1):
            f[ell] = _grow(f[ell + 1], edge_sets[ell + 1], n)
    return state, ob, f

def cut_predictor(n, L, r, ell_i, q_i, edge_sets, protected_edges):
    """Pure graph-reachability exposure predictor (r = 1 geometry).

    For each noise slot ell, the visible interaction cone is
      I(ell) = ob_cone(ell)  cap state_cone(ell)      (post-insertion)
      I(ell) = f_cone(ell)   cap state_cone(ell)      (pre-insertion)
    The slot is combinatorially protected when every 2-subset of I(ell) is
    a protected pair. 'All slots protected' is a sufficient condition for a
    vanishing response; the predictor reports exposure otherwise, which is
    a necessary condition only (cancellations can still protect).
    Returns {"protected": bool, "per_slot": {ell: (sorted I, exposed)}}.
    """
    if r != 1:
        raise ValueError("cut_predictor implemented for r = 1")
    prot = {frozenset(e) for e in protected_edges}
    state, ob, f = cones(n, L, r, ell_i, q_i, edge_sets)
    per_slot, all_ok = {}, True
    for ell in range(L):
        if ell >= ell_i:
            I = ob[ell] & state[ell]
        elif ell in f:
            I = f[ell] & state[ell]
        else:
            I = set()
        exposed = any(frozenset(p) not in prot
                      for p in combinations(sorted(I), 2))
        per_slot[ell] = (sorted(I), bool(exposed))
        all_ok = all_ok and not exposed
    return {"protected": bool(all_ok), "per_slot": per_slot}


# --------------------------------------------------- visible pair weights
def visible_pair_weights(model, theta, beta, ell_i, q_i, pairs):
    """Pairwise decomposition of the first-order response pairing.

    Post-insertion slots pair OB_slot with the derivative operator D_slot;
    pre-insertion slots pair the adjoint functional F_slot with the state.
    The weight of coherence pair p = (a, b) is the summed magnitude of the
    two factors' product on p, so a pair carries weight only where BOTH the
    functional and the propagated operator are supported: the quantitative
    form of the interaction cone I(ell). (Plumbing mirrors
    cohalign_rates.response_rates so the two share slot conventions.)
    """
    n, L = model.n, model.L
    Zq = embed_one_qubit(PAULI_Z, q_i, n)
    psi = model.initial_state()
    rho = np.outer(psi, psi.conj())
    rho_slot, rho_pre_ins = [], None
    for ell in range(L):
        Uz = model.z_layer(theta[ell])
        Ux = model.xy_layer(beta[ell], ell)
        rho_z = Uz @ rho @ Uz.conj().T
        if ell == ell_i:
            rho_pre_ins = rho_z
        rho = Ux @ rho_z @ Ux.conj().T
        rho_slot.append(rho)
    C = -0.5j * (Zq @ rho_pre_ins - rho_pre_ins @ Zq)
    Ux_i = model.xy_layer(beta[ell_i], ell_i)
    D = Ux_i @ C @ Ux_i.conj().T
    D_slot = {ell_i: D}
    for ell in range(ell_i + 1, L):
        Ul = model.layer_unitary(theta, beta, ell)
        D = Ul @ D @ Ul.conj().T
        D_slot[ell] = D
    OB_slot = {L - 1: model.readout}
    for ell in range(L - 2, -1, -1):
        Ul = model.layer_unitary(theta, beta, ell + 1)
        OB_slot[ell] = Ul.conj().T @ OB_slot[ell + 1] @ Ul
    OB_ins = Ux_i.conj().T @ OB_slot[ell_i] @ Ux_i
    F = -0.5j * (OB_ins @ Zq - Zq @ OB_ins)
    Uz_i = model.z_layer(theta[ell_i])
    F = Uz_i.conj().T @ F @ Uz_i
    F_slot = {}
    if ell_i >= 1:
        F_slot[ell_i - 1] = F
        for ell in range(ell_i - 2, -1, -1):
            Ul = model.layer_unitary(theta, beta, ell + 1)
            F_slot[ell] = Ul.conj().T @ F_slot[ell + 1] @ Ul
    w = np.zeros(len(pairs))
    for k, (a, b) in enumerate(pairs):
        tot = 0.0
        for ell in range(ell_i, L):
            tot += abs(OB_slot[ell][b, a] * D_slot[ell][a, b])
        for ell in F_slot:
            tot += abs(F_slot[ell][b, a] * rho_slot[ell][a, b])
        w[k] = tot
    return w

def unprotected_fraction(weights, mask_protected):
    tot = float(np.sum(weights))
    if tot <= 0:
        return float("nan")
    return float(np.sum(weights[~mask_protected]) / tot)


# ----------------------------------- support-cut (Cheeger-type) mode bound
def support_cut_bound(g_pairvec, rates, mask_protected,
                      tol=TOL_RATE_ZERO):
    """For a diagonal restricted generator, the mode rate is the
    rate-weighted mean over the mode's support,
        lambda_mode = sum_p rates_p |g_p|^2 / sum_p |g_p|^2,
    so it is bounded below by phi * r_min, where phi is the mode's
    Frobenius-weight fraction on unprotected pairs and r_min the smallest
    nonzero unprotected rate. Returns (bound, phi, r_min, lam_exact)."""
    g2 = np.abs(np.asarray(g_pairvec)) ** 2
    tot = float(g2.sum())
    if tot <= 0:
        return float("nan"), float("nan"), float("nan"), float("nan")
    rates = np.asarray(rates, dtype=float)
    phi = float(g2[~mask_protected].sum() / tot)
    un = rates[~mask_protected]
    un = un[un > tol]
    r_min = float(un.min()) if un.size else 0.0
    lam_exact = float(np.sum(rates * g2) / tot)
    return phi * r_min, phi, r_min, lam_exact


# ------------------------------------------------ slot-resolved evolution
def evolve_dm_slotwise(model, theta, beta, channel, gvec):
    """Density-matrix evolution with an independent noise strength per slot
    (gvec[ell] after layer ell). gvec = gamma * ones reproduces
    Brickwork.evolve_dm exactly."""
    psi = model.initial_state()
    rho = np.outer(psi, psi.conj())
    for ell in range(model.L):
        U = model.layer_unitary(theta, beta, ell)
        rho = U @ rho @ U.conj().T
        g = float(gvec[ell])
        if channel is not None and g > 0:
            rho = channel.apply(rho, g, model.n)
    return rho

def grad_slotwise(model, theta, beta, channel, gvec, ell, q,
                  shift=np.pi / 2):
    tp, tm = theta.copy(), theta.copy()
    tp[ell, q] += shift
    tm[ell, q] -= shift
    fp = model.output(evolve_dm_slotwise(model, tp, beta, channel, gvec))
    fm = model.output(evolve_dm_slotwise(model, tm, beta, channel, gvec))
    return 0.5 * (fp - fm)

def m2_slotwise(model, beta, channel, gvec, ell, q, theta_draws,
                shift=np.pi / 2):
    vals = [grad_slotwise(model, th, beta, channel, gvec, ell, q, shift) ** 2
            for th in theta_draws]
    return float(np.mean(vals))


# ------------------------------------------------------- spectral helpers
def vertex_graph_laplacian(rates, pairs, n, r):
    """Weighted graph on the sector basis states with edge weight equal to
    the pairwise decay rate; returns the ordinary graph Laplacian D - W.
    (An analysis object for clustering, distinct from the restricted
    generator itself, which is diagonal on ordered pairs.)"""
    sec = sector_basis(n, r)
    idx = {s: k for k, s in enumerate(sec)}
    K = len(sec)
    W = np.zeros((K, K))
    for rate, (a, b) in zip(rates, pairs):
        W[idx[a], idx[b]] = rate
    W = 0.5 * (W + W.T)
    return np.diag(W.sum(axis=1)) - W

def fiedler_value(Lap):
    ev = np.linalg.eigvalsh(Lap)
    return float(ev[1]) if len(ev) > 1 else float("nan")
'''
(PKG_DIR / "cgl_graph.py").write_text(CGL_GRAPH_SRC)
import cgl_graph as cgg
importlib.reload(cgg)
print(f"cgl_graph written and imported ({len(CGL_GRAPH_SRC.splitlines())} lines)")

# Phase A — self-test battery

Exact identities only; every test asserts. Failure here means the
environment or an edit broke the machinery, not that physics disagreed.


In [ ]:
out_csv_A = OUT_DIR / "phaseA_selftests.csv"
N, L, R = CFG["n"], CFG["L"], CFG["r"]
SUITE = cac.build_channel_suite(N)
MODEL = cac.Brickwork(N, L, R)
BETA = cac.teacher_background(TEACHER_SEED, L, N)

if cache_or_compute(out_csv_A, "Phase A"):
    print("=" * 72); print("PHASE A -- SELF-TEST BATTERY"); print("=" * 72)
    t0 = time.time(); tests = []
    def check(name, ok, detail=""):
        tests.append({"test": name, "passed": bool(ok), "detail": detail})
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}  {detail}")
        assert ok, f"self-test failed: {name} ({detail})"

    rng = np.random.default_rng(0)
    # 1. trace preservation of every dissipative channel on a random state
    A = rng.normal(size=(2**N, 2**N)) + 1j * rng.normal(size=(2**N, 2**N))
    rho = A @ A.conj().T; rho = rho / np.trace(rho)
    worst = 0.0
    for name, ch in SUITE.items():
        dev = abs(float(np.real(np.trace(ch.apply(rho, 0.05, N)))) - 1.0)
        worst = max(worst, dev)
    check("trace preservation (all channels)", worst < 1e-10, f"worst {worst:.1e}")

    # 2. sector preservation of the noiseless circuit
    theta = rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
    rho0 = MODEL.evolve_dm(theta, BETA)
    Pr = MODEL.P_r
    leak = float(np.real(np.trace(rho0))) - float(np.real(np.trace(Pr @ rho0 @ Pr)))
    check("sector preservation (noiseless circuit)", abs(leak) < 1e-10,
          f"leak {leak:.1e}")

    # 3. diagonality of the restricted generator, phase-covariant families
    diag_family = ("amp_damp", "dephase", "depol", "inhom_dephase",
                   "site_amp_damp", "corr_dephase", "coh_diss_mix")
    worst = 0.0
    for name in diag_family:
        L_r, pairs = car.restricted_generator(SUITE[name], N, R, GAMMA_PROBE)
        worst = max(worst, cgg.offdiag_mass(L_r))
    check("pair-basis diagonality (phase-covariant families)",
          worst < 1e-8, f"worst off-diag mass {worst:.1e}")

    # 4. analytic first-order rates against the finite-probe generator
    #    (agreement up to the documented O(gamma_probe) bias)
    worst_rel = 0.0
    for name in ("dephase", "amp_damp", "depol", "x_err", "biased_pauli",
                 "inhom_dephase", "site_amp_damp", "corr_dephase"):
        ch = SUITE[name]
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        num = cgg.numeric_diag_rates(L_r)
        ana = np.array([cgg.analytic_rate_pair(ch, a, b, N) for a, b in pairs])
        scale = max(float(ana.max()), 1e-12)
        rel = float(np.max(np.abs(num - ana)) / scale)
        worst_rel = max(worst_rel, rel)
    check("analytic rate identity (probe-bias tolerance)",
          worst_rel < 5 * GAMMA_PROBE * max(1.0, 2 * N),
          f"worst rel dev {worst_rel:.2e} at probe {GAMMA_PROBE:.0e}")

    # 5. difference-form (graph-Laplacian) certificate, dephasing family
    worst = max(cgg.check_difference_form(SUITE["dephase"], N, R),
                cgg.check_difference_form(SUITE["inhom_dephase"], N, R),
                cgg.check_difference_form(SUITE["corr_dephase"], N, R))
    check("difference-form identity (dephasing family)", worst < 1e-12,
          f"worst {worst:.1e}")

    # 6. potential-form (Schroedinger) certificate, damping family
    worst = max(cgg.check_potential_form(SUITE["amp_damp"], N, R),
                cgg.check_potential_form(SUITE["site_amp_damp"], N, R))
    check("potential-form identity (damping family)", worst < 1e-12,
          f"worst {worst:.1e}")

    # 7. correlated-control kernel = ordered protected pairs
    ch = SUITE["corr_dephase"]
    L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
    rates = cgg.numeric_diag_rates(L_r)
    mask = cgg.protected_mask(rates, tol=1e-6)
    kd = cgg.kernel_dimension(L_r, tol=1e-6)
    n_prot = int(mask.sum())
    n_edges = len(ch.edges)
    check("protection kernel dimension", kd == n_prot == 2 * n_edges,
          f"kernel {kd}, protected pairs {n_prot}, 2|E| {2*n_edges}")

    # 8. null coherent control audits identically to amplitude damping
    L_a, _ = car.restricted_generator(SUITE["amp_damp"], N, R, GAMMA_PROBE)
    L_c, _ = car.restricted_generator(SUITE["coh_diss_mix"], N, R, GAMMA_PROBE)
    dev = float(np.max(np.abs(np.real(np.diag(L_a)) - np.real(np.diag(L_c)))))
    check("coherent-dissipative null duplicates amplitude damping",
          dev < 1e-9, f"max diag dev {dev:.1e}")

    # 9. slotwise evolution reproduces uniform evolution exactly
    g = 0.03
    r1 = MODEL.evolve_dm(theta, BETA, SUITE["dephase"], g)
    r2 = cgg.evolve_dm_slotwise(MODEL, theta, BETA, SUITE["dephase"],
                                np.full(L, g))
    dev = float(np.max(np.abs(r1 - r2)))
    check("slotwise evolution consistency", dev < 1e-12, f"max dev {dev:.1e}")

    # 10. response frame invariance (df0 slot-independent)
    rr = car.response_rates(MODEL, theta, BETA, SUITE["dephase"], 1, 0,
                            GAMMA_PROBE, frame_check=True)
    check("response frame invariance", rr["frame_defect"] < 1e-8,
          f"defect {rr['frame_defect']:.1e}")

    # 11. cut predictor protection at the shallow n=8 geometry (guarded)
    if N == 8 and L == 3:
        cp = cgg.cut_predictor(N, L, R, 2, 1, MODEL.E,
                               SUITE["corr_dephase"].edges)
        check("cut predictor: shallow protection (param (2,1))",
              cp["protected"], str({k: v[0] for k, v in cp["per_slot"].items()}))

    # 12. support-cut bound holds on one worked case (dephasing, primary)
    L_d, pairs_d = car.restricted_generator(SUITE["dephase"], N, R,
                                            GAMMA_PROBE)
    rates_d = cgg.numeric_diag_rates(L_d)
    modes = car.slot_modes(MODEL, theta, BETA, 1, 0)
    gvec = car.vec_offdiag(modes[1], pairs_d)
    bound, phi, rmin, lam = cgg.support_cut_bound(
        gvec, rates_d, cgg.protected_mask(rates_d))
    check("support-cut bound (worked case)",
          lam >= bound - 1e-9, f"lam {lam:.4f} >= bound {bound:.4f}")

    pd.DataFrame(tests).to_csv(out_csv_A, index=False)
    write_meta(out_csv_A, "A")
    TIMINGS["A"] = time.time() - t0
    print(f"Phase A complete: {len(tests)} tests, {TIMINGS['A']:.1f}s "
          f"-> {out_csv_A.name}")

# Phase B — coherence-graph recovery

For each channel: diagonality certificate, numeric rates against the
analytic first-order formulas (agreement up to the documented probe bias),
worst-case rate against the analytic maximum, spectral anisotropy, kernel
dimension, protection components, and the exact structural form
certificate (difference or potential form at machine precision). The
correlated control must appear as a disconnected graph whose kernel equals
the ordered protected pairs; the signed coherent control must show a
vanishing Hermitian rate.


In [ ]:
out_csv_B = OUT_DIR / "phaseB_graph_recovery.csv"
out_csv_B2 = OUT_DIR / "phaseB_pair_rates.csv"
if cache_or_compute(out_csv_B, "Phase B"):
    print("=" * 72); print("PHASE B -- COHERENCE-GRAPH RECOVERY"); print("=" * 72)
    t0 = time.time(); rows, pair_rows = [], []
    diag_family = ("amp_damp", "dephase", "depol", "x_err", "biased_pauli",
                   "inhom_dephase", "site_amp_damp", "corr_dephase",
                   "coh_diss_mix")
    for name in diag_family:
        ch = SUITE[name]
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        num = cgg.numeric_diag_rates(L_r)
        ana = np.array([cgg.analytic_rate_pair(ch, a, b, N)
                        for (a, b) in pairs])
        ev = cgg.hermitian_rate_spectrum(L_r)
        lam_w = float(ev[-1])
        lam_ana = float(ana.max())
        mask = cgg.protected_mask(num, tol=1e-6)
        ncomp, sizes = cgg.protection_components(N, R, mask, pairs)
        # structural family label and the exact algebraic certificate
        if name in ("dephase", "inhom_dephase", "corr_dephase"):
            family, cert = "difference (Laplacian)", \
                cgg.check_difference_form(ch, N, R)
        elif name in ("amp_damp", "site_amp_damp", "coh_diss_mix"):
            family, cert = "potential (Schroedinger)", \
                cgg.check_potential_form(SUITE["amp_damp"]
                                         if name == "coh_diss_mix" else ch,
                                         N, R)
        else:
            family, cert = "affine in Hamming distance", float("nan")
        scale = max(lam_ana, 1e-12)
        rows.append({
            "channel": name, "structural_family": family,
            "offdiag_mass": cgg.offdiag_mass(L_r),
            "max_rel_dev_numeric_vs_analytic":
                float(np.max(np.abs(num - ana)) / scale),
            "lambda_worst_numeric": lam_w,
            "lambda_worst_analytic": lam_ana,
            "spectral_anisotropy": float((ev[-1] - ev[0]) / max(ev[-1], 1e-30)),
            "kernel_dim": cgg.kernel_dimension(L_r, tol=1e-6),
            "n_protected_pairs": int(mask.sum()),
            "protection_components": ncomp,
            "component_sizes": str(sizes[:6]),
            "form_certificate_maxdev": cert,
        })
        for rate_n, rate_a, (a, b) in zip(num, ana, pairs):
            pair_rows.append({"channel": name, "a": a, "b": b,
                              "rate_numeric": float(rate_n),
                              "rate_analytic": float(rate_a)})
        print(f"  {name:16s} {family:26s} lam {lam_w:7.4f} "
              f"(analytic {lam_ana:7.4f})  kernel {int(mask.sum()):2d}  "
              f"components {ncomp}")
    # signed pure coherent control: Hermitian rate must vanish
    L_s, _ = car.restricted_generator(cac.SiteZOverRotation(), N, R,
                                      GAMMA_PROBE)
    ev_s = cgg.hermitian_rate_spectrum(L_s)
    rows.append({"channel": "site_z_overrotation",
                 "structural_family": "pure coherent (signed)",
                 "offdiag_mass": cgg.offdiag_mass(L_s),
                 "max_rel_dev_numeric_vs_analytic": float(np.max(np.abs(ev_s))),
                 "lambda_worst_numeric": float(ev_s[-1]),
                 "lambda_worst_analytic": 0.0,
                 "spectral_anisotropy": float("nan"), "kernel_dim": np.nan,
                 "n_protected_pairs": np.nan, "protection_components": np.nan,
                 "component_sizes": "", "form_certificate_maxdev": float("nan")})
    pd.DataFrame(rows).to_csv(out_csv_B, index=False)
    pd.DataFrame(pair_rows).to_csv(out_csv_B2, index=False)
    write_meta(out_csv_B, "B", companions=[out_csv_B2.name])
    write_meta(out_csv_B2, "B")
    TIMINGS["B"] = time.time() - t0
    print(f"Phase B complete in {TIMINGS['B']:.1f}s -> {out_csv_B.name}")

# Phase C — the cut-crossing predictor of light-cone exposure

The headline experiment. For each depth in the sweep, the combinatorial
predictor (pure edge-set reachability, no linear algebra, no simulation)
declares each audited parameter protected or exposed against the
correlated control. Alongside it: the visible weight fraction on
unprotected pairs (the quantitative cut mass), the generator-level
response prediction $\omega^{\mathrm{resp}}$, and the CRN-measured
$\hat\omega$ at the small strengths. Binary agreement uses the 0.05
threshold on $|\hat\omega|$; the visible fraction is expected to track the
measured exposure quantitatively. Predicted exposure is necessary, not
sufficient, so any protected measurement under a predicted exposure is
reported, not suppressed. Counting unit: the agreement tally counts rows
of the output table, ten depth-parameter configurations at two strengths
each, under the fixed threshold $|\hat\omega| < 0.05$; it is not twenty
independent architectural validations, and the threshold is not a
confidence-interval criterion (Phases H2 and P carry the interval-based
statistics).


In [ ]:
out_csv_C = OUT_DIR / "phaseC_cut_crossing.csv"
if cache_or_compute(out_csv_C, "Phase C"):
    print("=" * 72)
    print("PHASE C -- CUT-CROSSING PREDICTOR OF LIGHT-CONE EXPOSURE")
    print("=" * 72)
    t0 = time.time(); rows = []
    ch = SUITE["corr_dephase"]
    for L_d in CFG["depths"]:
        model_d = cac.Brickwork(N, L_d, R)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        rates = cgg.numeric_diag_rates(L_r)
        mask = cgg.protected_mask(rates, tol=1e-6)
        lam_w = float(cgg.hermitian_rate_spectrum(L_r)[-1])
        act = cab.activity_preflight(model_d, beta_d,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        params = cab.pick_parameters(act, k=2, distinct_layers=True)
        # shallow depth: audit every active parameter as in the CohAlign
        # control; deeper: the preflight-selected locations
        if L_d == 3:
            flat = [(ell, q) for ell in range(L_d) for q in range(N)
                    if act[ell, q] > 1e-4 * act.max()]
            params = flat
        for (ell_i, q_i) in params:
            cp = cgg.cut_predictor(N, L_d, R, ell_i, q_i, model_d.E, ch.edges)
            # quantitative companion: visible weight on unprotected pairs
            rng = np.random.default_rng(EST_SEED)
            fracs = []
            for _ in range(CFG["n_theta_est"]):
                th = rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L_d, N))
                w = cgg.visible_pair_weights(model_d, th, beta_d,
                                             ell_i, q_i, pairs)
                f = cgg.unprotected_fraction(w, mask)
                if np.isfinite(f):
                    fracs.append(f)
            vis_frac = float(np.mean(fracs)) if fracs else float("nan")
            # response-susceptibility prediction (CohAlign machinery)
            resp = car.lambda_vis_resp(model_d, beta_d, ch, ell_i, q_i,
                                       n_theta=CFG["n_theta_est"],
                                       delta_init=DELTA_INIT, seed=EST_SEED,
                                       gamma_probe=GAMMA_PROBE,
                                       bootstrap_B=CFG["bootstrap_B"],
                                       bootstrap_seed=BOOT_SEED)
            omega_pred = resp["rate_sum"] / (L_d * lam_w)
            # CRN paired measurement at the small strengths
            for g in CFG["gamma_meas"]:
                meas = cab.paired_degradation(model_d, beta_d, ell_i, q_i,
                                              ch, g,
                                              n_theta=CFG["n_theta_meas"],
                                              delta_init=DELTA_INIT,
                                              seed=MEAS_SEED, shift=SHIFT,
                                              bootstrap_B=CFG["bootstrap_B"],
                                              boot_seed=BOOT_SEED)
                om_hat = cab.omega_hat(meas["delta_tilde"], g, L_d, lam_w)
                rows.append({
                    "depth": L_d, "ell": ell_i, "q": q_i, "gamma": g,
                    "cut_predictor_protected": cp["protected"],
                    "cut_slots": str({k: v[0]
                                      for k, v in cp["per_slot"].items()}),
                    "visible_unprotected_fraction": vis_frac,
                    "omega_resp_pred": float(omega_pred),
                    "omega_hat_measured": float(om_hat),
                    "delta_tilde": meas["delta_tilde"],
                    "delta_ci_lo": meas["ci_lo"], "delta_ci_hi": meas["ci_hi"],
                    "n_used": meas["n_used"], "ess": resp["ess"],
                })
                print(f"  L={L_d} ({ell_i},{q_i}) g={g:.2f}  "
                      f"cut={'PROT' if cp['protected'] else 'EXP '}  "
                      f"vis_frac={vis_frac:.3f}  pred={omega_pred:+.3f}  "
                      f"meas={om_hat:+.3f}")
    df = pd.DataFrame(rows)
    df["binary_agree"] = ((df.cut_predictor_protected &
                           (df.omega_hat_measured.abs() < 0.05)) |
                          (~df.cut_predictor_protected &
                           (df.omega_hat_measured.abs() >= 0.05)))
    df.to_csv(out_csv_C, index=False)
    write_meta(out_csv_C, "C")
    TIMINGS["C"] = time.time() - t0
    n_ok = int(df.binary_agree.sum())
    print(f"Phase C complete in {TIMINGS['C']:.1f}s: binary agreement "
          f"{n_ok}/{len(df)} -> {out_csv_C.name}")

# Phase D — support bound and adversarial sweep

Part 1 verifies the weighted-average support bound $\lambda_{\mathrm{mode}} \ge \phi\, r_{\min}$ across the
diagonal channel family and every active parameter at the primary depth,
with shared estimator draws. Part 2 is deliberately adversarial: random
inhomogeneous dephasing profiles and random ZZ edge-set channels (random
covariant generators), auditing whether $\omega^{\mathrm{resp}} \in [0,1]$
and whether the empirical ordering $\omega^{\mathrm{mode}} \ge
\omega^{\mathrm{resp}}$ survives. Either outcome is informative; a
violation would be a sharp structural finding and is recorded, not
filtered.


In [ ]:
out_csv_D = OUT_DIR / "phaseD_bound.csv"
out_csv_D2 = OUT_DIR / "phaseD_sweep.csv"
if cache_or_compute(out_csv_D, "Phase D"):
    print("=" * 72)
    print("PHASE D -- SUPPORT-CUT BOUND AND ADVERSARIAL SWEEP")
    print("=" * 72)
    t0 = time.time()
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    PARAMS = [(ell, q) for ell in range(L) for q in range(N)
              if act[ell, q] > 1e-4 * act.max()]
    print(f"  active parameters at L={L}: {PARAMS}")

    # -- Part 1: bound audit on the diagonal-generator channels -----------
    rows = []
    diag_channels = ("amp_damp", "dephase", "depol", "inhom_dephase",
                     "site_amp_damp", "corr_dephase")
    rng = np.random.default_rng(EST_SEED)
    theta_draws = [rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                   for _ in range(CFG["n_theta_est"])]
    for name in diag_channels:
        ch = SUITE[name]
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        rates = cgg.numeric_diag_rates(L_r)
        mask = cgg.protected_mask(rates, tol=1e-6)
        lam_w = float(cgg.hermitian_rate_spectrum(L_r)[-1])
        for (ell_i, q_i) in PARAMS:
            lam_vals, bounds, phis = [], [], []
            for th in theta_draws:
                for ell, M in car.slot_modes(MODEL, th, BETA,
                                             ell_i, q_i).items():
                    g = car.vec_offdiag(M, pairs)
                    b, phi, rmin, lam = cgg.support_cut_bound(g, rates, mask)
                    if np.isfinite(lam):
                        lam_vals.append(lam); bounds.append(b); phis.append(phi)
            lam_mode = float(np.mean(lam_vals))
            bound = float(np.mean(bounds))
            rows.append({"channel": name, "ell": ell_i, "q": q_i,
                         "lambda_mode": lam_mode,
                         "support_cut_bound": bound,
                         "phi_unprotected": float(np.mean(phis)),
                         "lambda_coh": lam_w,
                         "bound_holds": bool(lam_mode >= bound - 1e-9)})
    df1 = pd.DataFrame(rows)
    df1.to_csv(out_csv_D, index=False)
    n_ok = int(df1.bound_holds.sum())
    print(f"  bound audit: holds on {n_ok}/{len(df1)} channel-parameter "
          f"pairs (mean slack "
          f"{float((df1.lambda_mode - df1.support_cut_bound).mean()):.3f})")

    # -- Part 2: adversarial sweep over random covariant generators -------
    (ELL_A, Q_A) = PARAMS[0]
    sweep_rows = []
    srng = np.random.default_rng(SWEEP_SEED)
    def audit_random(ch, kind, k):
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        lam_w = float(cgg.hermitian_rate_spectrum(L_r)[-1])
        op = car.lambda_mode(MODEL, BETA, ch, ELL_A, Q_A,
                             n_theta=CFG["n_theta_est"],
                             delta_init=DELTA_INIT, seed=EST_SEED,
                             gamma_probe=GAMMA_PROBE, L_r=L_r, pairs=pairs)
        resp = car.lambda_vis_resp(MODEL, BETA, ch, ELL_A, Q_A,
                                   n_theta=CFG["n_theta_est"],
                                   delta_init=DELTA_INIT, seed=EST_SEED,
                                   gamma_probe=GAMMA_PROBE,
                                   bootstrap_B=50, bootstrap_seed=BOOT_SEED)
        om_mode = op["lambda_vis_op"] / lam_w
        om_resp = resp["rate_sum"] / (L * lam_w)
        sweep_rows.append({"kind": kind, "index": k,
                           "lambda_coh": lam_w,
                           "omega_mode": float(om_mode),
                           "omega_resp": float(om_resp),
                           "in_unit_interval":
                               bool(-1e-6 <= om_resp <= 1 + 1e-6),
                           "mode_ge_resp":
                               bool(om_mode >= om_resp - 1e-6)})
    for k in range(CFG["sweep_profiles"]):
        w = srng.uniform(0.2, 2.0, size=N)
        audit_random(cac.InhomogeneousDephasing(w), "dephasing_profile", k)
    ring = [(j, (j + 1) % N) for j in range(N)]
    for k in range(CFG["sweep_edge_sets"]):
        m = int(srng.integers(1, N))
        sel = srng.choice(len(ring), size=m, replace=False)
        edges = [ring[int(s)] for s in sel]
        audit_random(cac.CorrelatedDephasing(N, edges=edges),
                     "zz_edge_set", k)
    df2 = pd.DataFrame(sweep_rows)
    df2.to_csv(out_csv_D2, index=False)
    write_meta(out_csv_D, "D", companions=[out_csv_D2.name])
    write_meta(out_csv_D2, "D")
    TIMINGS["D"] = time.time() - t0
    print(f"  sweep: {len(df2)} random covariant generators; "
          f"omega_resp in [0,1] on {int(df2.in_unit_interval.sum())}, "
          f"mode >= resp on {int(df2.mode_ge_resp.sum())}")
    print(f"Phase D complete in {TIMINGS['D']:.1f}s -> {out_csv_D.name}, "
          f"{out_csv_D2.name}")

# Phase E — heat-kernel slot closed forms

The closed-form targets (2/3 at $L=3$ and 1/2 at $L=4$ for isotropic
dephasing, 1 for amplitude damping, final slot pairing to zero) are
recorded in the phase source **before** any measurement, serving as the
dated analysis plan. The phase then computes the slot-resolved response
rates and reports deviations from target.


In [ ]:
out_csv_E = OUT_DIR / "phaseE_slot_forms.csv"
if cache_or_compute(out_csv_E, "Phase E"):
    print("=" * 72)
    print("PHASE E -- HEAT-KERNEL SLOT CLOSED FORMS (PRE-REGISTERED)")
    print("=" * 72)
    t0 = time.time(); rows = []
    # Pre-registered closed-form targets, PINNED to their reference
    # parameters (v1.5 audit correction: an automatically selected
    # parameter must not silently change the mathematical subject of an
    # exact test; the depth-four value 1/2 is a property of parameter
    # (2,0), not of the depth, and e.g. (2,7) has population alignment
    # 0.4861981920 under the same prior). Targets keyed by the full
    # channel-depth-parameter configuration; the preflight is used only
    # to confirm the pinned parameter is active, never to select.
    TARGETS = {("dephase", 3, (2, 1)): 2.0 / 3.0,
               ("dephase", 4, (2, 0)): 0.5,
               ("amp_damp", 3, (2, 1)): 1.0}
    for (name, L_d, (ell_pin, q_pin)), target in TARGETS.items():
        ch = SUITE[name]
        model_d = cac.Brickwork(N, L_d, R)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        lam_w = float(cgg.hermitian_rate_spectrum(L_r)[-1])
        act = cab.activity_preflight(model_d, beta_d,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        (ell_i, q_i) = (ell_pin, q_pin)
        assert act[ell_i, q_i] > 1e-4 * act.max(),             f"pinned reference parameter ({ell_i},{q_i}) inactive at L={L_d}"
        resp = car.lambda_vis_resp(model_d, beta_d, ch, ell_i, q_i,
                                   n_theta=CFG["n_theta_est"],
                                   delta_init=DELTA_INIT, seed=EST_SEED,
                                   gamma_probe=GAMMA_PROBE,
                                   bootstrap_B=CFG["bootstrap_B"],
                                   bootstrap_seed=BOOT_SEED)
        per = resp["per_slot"]
        omega = resp["rate_sum"] / (L_d * lam_w)
        final_ratio = per[L_d - 1] / lam_w
        rows.append({"channel": name, "depth": L_d, "ell": ell_i, "q": q_i,
                     "lambda_coh": lam_w,
                     "per_slot_over_lambda":
                         str({k: round(v / lam_w, 4)
                              for k, v in sorted(per.items())}),
                     "final_slot_ratio": float(final_ratio),
                     "omega_resp": float(omega),
                     "closed_form_target": float(target),
                     "abs_dev_from_target": float(abs(omega - target))})
        print(f"  {name:9s} L={L_d} ({ell_i},{q_i}): "
              f"final slot {final_ratio:+.2e}, omega {omega:.4f} "
              f"(target {target:.4f})")
        assert abs(omega - target) < 1e-8,             f"exact identity failed at pinned ({ell_i},{q_i}), L={L_d}"
    pd.DataFrame(rows).to_csv(out_csv_E, index=False)
    write_meta(out_csv_E, "E")
    TIMINGS["E"] = time.time() - t0
    print(f"Phase E complete in {TIMINGS['E']:.1f}s -> {out_csv_E.name}")

# Phase G — two-excitation coherence-graph spectra

The analytic rate formulas extend without modification to $r = 2$ (pair
dimension 756 at $n = 8$), where the correlated control's protected
structure collapses. The phase computes protected fractions, protection
components and Fiedler values of the vertex distinguishability graph per
channel, verifies the analytic rates numerically for the configured
channels, and records the $r = 1$ protected fraction of the correlated
control as the comparison line.


In [ ]:
out_csv_G = OUT_DIR / "phaseG_r2_graph.csv"
if cache_or_compute(out_csv_G, "Phase G"):
    print("=" * 72)
    print("PHASE G -- TWO-EXCITATION COHERENCE-GRAPH SPECTRA")
    print("=" * 72)
    t0 = time.time(); rows = []
    R2 = 2
    pairs2 = cac.pair_basis(N, R2)
    print(f"  sector dimension {len(cac.sector_basis(N, R2))}, "
          f"pair dimension {len(pairs2)}")
    graph_family = ("dephase", "inhom_dephase", "corr_dephase",
                    "amp_damp", "site_amp_damp", "depol")
    for name in graph_family:
        ch = SUITE[name]
        ana = np.array([cgg.analytic_rate_pair(ch, a, b, N)
                        for (a, b) in pairs2])
        mask = cgg.protected_mask(ana, tol=1e-9)
        ncomp, sizes = cgg.protection_components(N, R2, mask, pairs2)
        Lap = cgg.vertex_graph_laplacian(ana, pairs2, N, R2)
        fied = cgg.fiedler_value(Lap)
        row = {"channel": name,
               "pair_dim": len(pairs2),
               "protected_pairs": int(mask.sum()),
               "protected_fraction": float(mask.mean()),
               "protection_components": ncomp,
               "component_sizes": str(sizes[:8]),
               "fiedler_value": fied,
               "rate_min": float(ana.min()), "rate_max": float(ana.max()),
               "numeric_checked": name in CFG["r2_numeric_channels"],
               "numeric_max_rel_dev": float("nan")}
        if name in CFG["r2_numeric_channels"]:
            L_r2, p2 = car.restricted_generator(ch, N, R2, GAMMA_PROBE)
            num = cgg.numeric_diag_rates(L_r2)
            scale = max(float(ana.max()), 1e-12)
            row["numeric_max_rel_dev"] = \
                float(np.max(np.abs(num - ana)) / scale)
        rows.append(row)
        print(f"  {name:16s} protected {int(mask.sum()):4d}/{len(pairs2)} "
              f"({100*mask.mean():5.2f}%)  components {ncomp:2d}  "
              f"Fiedler {fied:8.3f}"
              + (f"  numeric dev {row['numeric_max_rel_dev']:.1e}"
                 if row["numeric_checked"] else ""))
    # the r=1 comparison line for the protection-collapse figure
    ch = SUITE["corr_dephase"]
    ana1 = np.array([cgg.analytic_rate_pair(ch, a, b, N)
                     for (a, b) in cac.pair_basis(N, 1)])
    rows.append({"channel": "corr_dephase_r1_reference",
                 "pair_dim": len(ana1),
                 "protected_pairs": int((ana1 < 1e-9).sum()),
                 "protected_fraction": float((ana1 < 1e-9).mean()),
                 "protection_components": np.nan, "component_sizes": "",
                 "fiedler_value": np.nan,
                 "rate_min": float(ana1.min()), "rate_max": float(ana1.max()),
                 "numeric_checked": False,
                 "numeric_max_rel_dev": float("nan")})
    pd.DataFrame(rows).to_csv(out_csv_G, index=False)
    write_meta(out_csv_G, "G")
    TIMINGS["G"] = time.time() - t0
    print(f"Phase G complete in {TIMINGS['G']:.1f}s -> {out_csv_G.name}")

# Phase H — deep-circuit measurements at high ensembles

The one depth at which the prediction and the measurement part company is
also, in Phase C, the least measured, so this phase carries the burden of
proof properly:

* **Prediction side.** The response susceptibility at each audited
  parameter is re-estimated with 120 draws, and its weighted-bootstrap
  95% interval is recorded rather than discarded, since at depth six the
  effective sample size of the estimator is itself small.
* **Measurement side.** 400 CRN paired draws per point at three strengths
  $\gamma \in \{0.005, 0.01, 0.02\}$; the first-order limit is the
  intercept of the exact quadratic through the three alignment estimates,
  with a joint bootstrap over the shared draw indices giving its 95%
  interval, and the linear intercept through the two smallest strengths
  recorded as the robustness companion.
* **Control.** Depth five runs through the identical machinery; agreement
  there certifies the pipeline unbiased at the adjacent depth.

This phase supplies the raw deep-circuit measurements. The decision rule
applied to the depth-six question in this pipeline is the difference-based
three-way analysis of Phase H2; the overlap-based verdict column that this
phase writes is retained in the CSV for completeness and is not quoted.
Smoke mode exercises the identical code path at reduced ensembles; its
verdict columns are indicative only.


In [ ]:
out_csv_H = OUT_DIR / "phaseH_deep_stress.csv"
if cache_or_compute(out_csv_H, "Phase H"):
    print("=" * 72)
    print("PHASE H -- DEEP-CIRCUIT STRESS TEST AT HIGH ENSEMBLES")
    print("=" * 72)
    t0 = time.time()
    ch = SUITE["corr_dephase"]
    L_rH, pairsH = car.restricted_generator(ch, N, R, GAMMA_PROBE)
    lam_H = float(cgg.hermitian_rate_spectrum(L_rH)[-1])
    stress_top = max(CFG["stress_depths"])
    rows = []
    for L_d in CFG["stress_depths"]:
        role = "stress" if L_d == stress_top else "control"
        model_d = cac.Brickwork(N, L_d, R)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        act = cab.activity_preflight(model_d, beta_d,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        params = cab.pick_parameters(act, k=2, distinct_layers=True)
        for (ell_i, q_i) in params:
            # prediction side: enlarged estimator ensemble WITH its own
            # bootstrap interval (recorded, not discarded)
            resp = car.lambda_vis_resp(model_d, beta_d, ch, ell_i, q_i,
                                       n_theta=CFG["stress_est"],
                                       delta_init=DELTA_INIT, seed=EST_SEED,
                                       gamma_probe=GAMMA_PROBE,
                                       bootstrap_B=CFG["bootstrap_B"],
                                       bootstrap_seed=BOOT_SEED)
            om_pred = resp["rate_sum"] / (L_d * lam_H)
            p_lo = resp["rate_sum_ci"][0] / (L_d * lam_H)
            p_hi = resp["rate_sum_ci"][1] / (L_d * lam_H)
            # measurement side: three strengths, CRN-aligned draws so a
            # JOINT bootstrap of the extrapolation intercept is exact
            gs = np.array(CFG["gamma_stress"], dtype=float)
            meas, gn_by_g, g0_ref = {}, {}, None
            for g in gs:
                m = cab.paired_degradation(model_d, beta_d, ell_i, q_i,
                                           ch, float(g),
                                           n_theta=CFG["stress_meas"],
                                           delta_init=DELTA_INIT,
                                           seed=MEAS_SEED, shift=SHIFT,
                                           bootstrap_B=CFG["bootstrap_B"],
                                           boot_seed=BOOT_SEED,
                                           return_draws=True)
                meas[float(g)] = m
                gn_by_g[float(g)] = np.asarray(m["gn_sq"])
                if g0_ref is None:
                    g0_ref = np.asarray(m["g0_sq"])
                else:
                    assert len(g0_ref) == len(m["g0_sq"]), \
                        "CRN draw alignment lost across strengths"
            om_hat = {g: cab.omega_hat(meas[g]["delta_tilde"], g, L_d, lam_H)
                      for g in map(float, gs)}
            # gamma -> 0: quadratic through the three point estimates
            # (exact interpolation), with a linear two-smallest-strength
            # intercept as the robustness companion
            coef = np.polyfit(gs, [om_hat[float(g)] for g in gs], 2)
            om_extrap = float(np.polyval(coef, 0.0))
            g2s = gs[np.argsort(gs)][:2]
            lin = np.polyfit(g2s, [om_hat[float(g)] for g in g2s], 1)
            om_extrap_lin = float(np.polyval(lin, 0.0))
            # joint bootstrap of the intercept over shared draw indices
            brng = np.random.default_rng(BOOT_SEED + 100)
            nA = len(g0_ref)
            boots = []
            for _ in range(CFG["bootstrap_B"]):
                idx = brng.integers(0, nA, size=nA)
                m0 = float(np.mean(g0_ref[idx]))
                if m0 <= 0:
                    continue
                oms = [(1.0 - float(np.mean(gn_by_g[float(g)][idx])) / m0)
                       / (2 * g * L_d * lam_H) for g in gs]
                boots.append(float(np.polyval(np.polyfit(gs, oms, 2), 0.0)))
            e_lo, e_hi = (np.percentile(boots, [2.5, 97.5]) if boots
                          else (float("nan"),) * 2)
            # pre-registered equivalence margin: intervals count as
            # disjoint only when separated by more than EQUIV_TOL in
            # omega units, an order of magnitude below the Phase C
            # discrepancy, so degenerate zero-width bootstrap intervals
            # cannot manufacture a disagreement
            EQUIV_TOL = 0.01
            gap = float(max(e_lo - p_hi, p_lo - e_hi))
            disjoint = bool(gap > EQUIV_TOL)
            if role == "control":
                verdict = "control_agrees" if not disjoint \
                    else "control_disagrees"
            else:
                verdict = "established" if disjoint else "resolved"
            for g in map(float, gs):
                m = meas[g]
                rows.append({
                    "depth": L_d, "ell": ell_i, "q": q_i, "role": role,
                    "gamma": g, "lambda_coh": lam_H,
                    "om_hat": om_hat[g],
                    "om_hat_lo": m["ci_lo"] / (2 * g * L_d * lam_H),
                    "om_hat_hi": m["ci_hi"] / (2 * g * L_d * lam_H),
                    "n_used": m["n_used"],
                    "omega_pred": float(om_pred),
                    "pred_lo": float(p_lo), "pred_hi": float(p_hi),
                    "ess_pred": resp["ess"],
                    "om_extrap": om_extrap,
                    "extrap_lo": float(e_lo), "extrap_hi": float(e_hi),
                    "om_extrap_lin": om_extrap_lin,
                    "interval_gap": gap,
                    "intervals_disjoint": disjoint, "verdict": verdict,
                })
            print(f"  L={L_d} ({ell_i},{q_i}) [{role:7s}]  "
                  f"pred {om_pred:.3f} [{p_lo:.3f},{p_hi:.3f}]  "
                  f"extrap {om_extrap:.3f} [{e_lo:.3f},{e_hi:.3f}]  "
                  f"(lin {om_extrap_lin:.3f})  -> {verdict}")
    pd.DataFrame(rows).to_csv(out_csv_H, index=False)
    write_meta(out_csv_H, "H")
    TIMINGS["H"] = time.time() - t0
    print(f"Phase H complete in {TIMINGS['H']:.1f}s -> {out_csv_H.name}")

# Phase I — pair-diagonality scope by channel and sector

Finite-probe pair mixing is channel-specific and can appear already at
$r = 1$: the X-error channel's two-site swap term maps $|1\rangle\langle
2|$ to $|2\rangle\langle 1|$ with coefficient $\gamma(1-\gamma)^{n-2}$
in the restricted operator, the biased-Pauli channel shows the same
structure, and depolarising noise gives zero for this term at $r = 1$
and mixes only at $r \ge 2$. This phase audits every configured channel at
both sectors and two probe strengths, checks the analytic swap predictions,
and states the scope per family: Z-diagonal and damping channels exactly
pair-diagonal at both sectors, X-bearing channels mixing at $r = 1$
already, depolarising at $r \ge 2$, with all mixing vanishing linearly in
the probe.


In [ ]:
out_csv_I2 = OUT_DIR / "phaseI2_diagonality_scope.csv"
if cache_or_compute(out_csv_I2, "Phase I"):
    print("=" * 72)
    print("PHASE I -- PAIR-DIAGONALITY SCOPE BY CHANNEL AND SECTOR")
    print("=" * 72)
    t0 = time.time(); rows = []
    # v1.3 correction: the audit exhibited r = 1 finite-probe mixing for
    # the X-error channel (the two-site swap term, coefficient
    # gamma (1-gamma)^{n-2} in the restricted operator) and for the
    # biased-Pauli channel; the earlier blanket statement that mixing is
    # absent at r = 1 was false. The scope is stated per channel and per
    # sector, with the analytic swap-term prediction checked at r = 1.
    for R_s in (1, 2):
        pairsS = cac.pair_basis(N, R_s)
        pidx = {p: k for k, p in enumerate(pairsS)}
        for name in CFG["r2_offdiag_channels"]:
            ch = SUITE[name]
            for gp in CFG["offdiag_probes"]:
                L_rS, _ = car.restricted_generator(ch, N, R_s, gp)
                A = np.abs(L_rS.copy()); np.fill_diagonal(A, 0.0)
                max_off = float(A.max())
                row = {"sector_r": R_s, "channel": name, "gamma_probe": gp,
                       "offdiag_mass": cgg.offdiag_mass(L_rS),
                       "max_abs_offdiag": max_off,
                       "max_abs_offdiag_over_probe": max_off / gp,
                       "swap_entry": float("nan"),
                       "swap_prediction": float("nan")}
                if R_s == 1 and (1, 2) in pidx:
                    k1, k2 = pidx[(1, 2)], pidx[(2, 1)]
                    row["swap_entry"] = float(max(abs(L_rS[k1, k2]),
                                                  abs(L_rS[k2, k1])))
                    if name == "x_err":
                        row["swap_prediction"] = gp * (1 - gp) ** (N - 2)
                if R_s == 2 and name == "depol" and (3, 5) in pidx:
                    k1, k2 = pidx[(3, 5)], pidx[(10, 12)]
                    row["swap_entry"] = float(max(abs(L_rS[k1, k2]),
                                                  abs(L_rS[k2, k1])))
                    row["swap_prediction"] = 4.0 * gp / 9.0
                rows.append(row)
        print(f"  sector r={R_s} audited "
              f"({len(CFG['r2_offdiag_channels'])} channels)")
    df = pd.DataFrame(rows)
    for (rr, name), g in df.groupby(["sector_r", "channel"]):
        g = g.sort_values("gamma_probe")
        if len(g) >= 2 and g.max_abs_offdiag.iloc[-1] > 1e-12:
            expo = float(np.log(g.max_abs_offdiag.iloc[-1]
                                / g.max_abs_offdiag.iloc[0])
                         / np.log(g.gamma_probe.iloc[-1]
                                  / g.gamma_probe.iloc[0]))
        else:
            expo = float("nan")
        df.loc[(df.sector_r == rr) & (df.channel == name),
               "probe_scaling_exponent"] = expo
    df.to_csv(out_csv_I2, index=False)
    write_meta(out_csv_I2, "I")
    TIMINGS["I"] = time.time() - t0
    hi = df[(df.gamma_probe == max(CFG["offdiag_probes"]))]
    for _, r in hi.iterrows():
        if r.max_abs_offdiag > 1e-12:
            print(f"  r={int(r.sector_r)} {r.channel:13s}: max off-diag "
                  f"{r.max_abs_offdiag:.3e} (probe exponent "
                  f"{r.probe_scaling_exponent:.3f})"
                  + (f", swap entry {r.swap_entry:.6e} vs prediction "
                     f"{r.swap_prediction:.6e}"
                     if np.isfinite(r.swap_prediction) else ""))
    print(f"Phase I complete in {TIMINGS['I']:.1f}s -> {out_csv_I2.name}")
    print("  Scope: finite-probe mixing is CHANNEL-SPECIFIC, present")
    print("  already at r = 1 for the X-bearing channels (swap terms) and")
    print("  at r >= 2 for depolarising noise; the Z-diagonal and damping")
    print("  families are exactly pair-diagonal at both sectors, and all")
    print("  mixing vanishes linearly in the probe strength.")

# Phase F2 — exact slot product laws

At the audited geometry the measured degradation curves are exact
elementary polynomials, $(1-2\gamma)^8$ for dephasing and $(1-\gamma)^6$
for damping, so a finite-difference Hessian approximates known closed
forms with an $O(h)$ bias, and the squared-generator (biharmonic) reading
does not hold for the probability parameterisation. This phase states the
exact result: per-slot integer exponents recovered from single-slot
strengths, the full product law verified on random per-slot strength
vectors to machine precision, closed-form
$\Lambda_1 = \alpha K/2$ and $K_2 = \alpha^2 K(K-1)/2$ with the exact
Hessian, and a step-size convergence study showing finite-difference
estimates converging to these values as $h \to 0$.


In [ ]:
out_csv_F2 = OUT_DIR / "phaseF2_product_law.csv"
out_csv_F2b = OUT_DIR / "phaseF2_fd_convergence.csv"
out_csv_F2c = OUT_DIR / "phaseF2_uniform_grid.csv"
if cache_or_compute(out_csv_F2, "Phase F2"):
    print("=" * 72)
    print("PHASE F2 -- EXACT SLOT PRODUCT LAWS (SUPERSEDES THE v1.1 FIT)")
    print("=" * 72)
    t0 = time.time()
    hrng = np.random.default_rng(HESS_SEED)
    theta_draws = [hrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                   for _ in range(CFG["n_theta_hess"])]
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    (ELL_F, Q_F) = cab.pick_parameters(act, k=1)[0]
    print(f"  parameter ({ELL_F},{Q_F}); {len(theta_draws)} shared CRN draws")
    ALPHA = {"dephase": 2.0, "amp_damp": 1.0, "corr_dephase": 2.0}
    rows, fd_rows, grid_rows = [], [], []
    g1, g2 = CFG["f2_slot_strengths"]
    prng = np.random.default_rng(SWEEP_SEED + 7)
    for name in ("dephase", "amp_damp", "corr_dephase"):
        ch = SUITE[name]
        alpha = ALPHA[name]
        def M2(gvec):
            return cgg.m2_slotwise(MODEL, BETA, ch, gvec, ELL_F, Q_F,
                                   theta_draws, SHIFT)
        m0 = M2(np.zeros(L))
        e = np.eye(L)
        # 1. integer slot exponents from a single strength, confirmed at a
        #    second strength: ratio(g e_l) = (1 - alpha g)^{k_l} exactly
        ks, dev_single = [], 0.0
        for ell in range(L):
            r1 = M2(g1 * e[ell]) / m0
            k = 0 if abs(r1 - 1.0) < 1e-12 else \
                int(round(np.log(r1) / np.log(1.0 - alpha * g1)))
            ks.append(k)
            r2 = M2(g2 * e[ell]) / m0
            dev_single = max(dev_single,
                             abs(r1 - (1 - alpha * g1) ** k),
                             abs(r2 - (1 - alpha * g2) ** k))
        # 2. full product law on random per-slot strength vectors
        dev_prod = 0.0
        for _ in range(CFG["f2_product_draws"]):
            gv = prng.uniform(0.0, 0.25, size=L)
            pred = float(np.prod([(1 - alpha * gv[l]) ** ks[l]
                                  for l in range(L)]))
            dev_prod = max(dev_prod, abs(M2(gv) / m0 - pred))
        # 3. exact coefficients and exact Hessian from the product law
        K = int(sum(ks))
        Lam1 = alpha * K / 2.0
        K2 = alpha ** 2 * K * (K - 1) / 2.0
        H = np.zeros((L, L))
        for l in range(L):
            H[l, l] = ks[l] * (ks[l] - 1) * alpha ** 2
            for m in range(L):
                if m != l:
                    H[l, m] = ks[l] * ks[m] * alpha ** 2
        same, cross = float(np.trace(H)), float(H.sum() - np.trace(H))
        rows.append({"channel": name, "ell": ELL_F, "q": Q_F,
                     "alpha": alpha, "slot_exponents": str(ks),
                     "K_total": K, "Lambda1_exact": Lam1, "K2_exact": K2,
                     "H_same_slot_exact": same, "H_cross_slot_exact": cross,
                     "single_slot_maxdev": dev_single,
                     "product_law_maxdev": dev_prod})
        print(f"  {name:14s} exponents {ks}  exact: Lambda1={Lam1:.0f} "
              f"K2={K2:.0f}  law dev single {dev_single:.1e} "
              f"product {dev_prod:.1e}")
        # 3b. uniform-strength verification grid for the figure
        for g in CFG["so2_gl_grid"]:
            meas = M2(np.full(L, float(g))) / m0
            exact = float((1 - alpha * g) ** K)
            grid_rows.append({"channel": name, "gamma": float(g),
                              "ratio_measured": float(meas),
                              "ratio_exact": exact,
                              "abs_dev": abs(float(meas) - exact)})
        # 4. step-size convergence of the v1.1 finite-difference Hessian,
        #    explaining its bias (linear in h for forward differences)
        if K > 0:
            for h in CFG["f2_fd_steps"]:
                Hh = np.zeros((L, L))
                r1v = {l: M2(h * e[l]) / m0 for l in range(L)}
                for l in range(L):
                    Hh[l, l] = (M2(2 * h * e[l]) / m0
                                - 2 * r1v[l] + 1.0) / h ** 2
                for l in range(L):
                    for m in range(l + 1, L):
                        rlm = M2(h * (e[l] + e[m])) / m0
                        Hh[l, m] = Hh[m, l] = \
                            (rlm - r1v[l] - r1v[m] + 1.0) / h ** 2
                K2h = 0.5 * float(Hh.sum())
                fd_rows.append({"channel": name, "h": h, "K2_fd": K2h,
                                "K2_exact": K2,
                                "abs_error": abs(K2h - K2),
                                "rel_error": abs(K2h - K2) / K2})
    pd.DataFrame(rows).to_csv(out_csv_F2, index=False)
    pd.DataFrame(fd_rows).to_csv(out_csv_F2b, index=False)
    write_meta(out_csv_F2b, "F2")
    pd.DataFrame(grid_rows).to_csv(out_csv_F2c, index=False)
    write_meta(out_csv_F2c, "F2")
    write_meta(out_csv_F2, "F2", companions=[out_csv_F2b.name, out_csv_F2c.name])
    TIMINGS["F2"] = time.time() - t0
    if fd_rows:
        d = pd.DataFrame(fd_rows)
        for name, g in d.groupby("channel"):
            g = g.sort_values("h")
            print(f"  FD convergence {name}: K2_hat "
                  f"{[round(v,2) for v in g.K2_fd]} at h {list(g.h)} "
                  f"-> exact {g.K2_exact.iloc[0]:.0f}")
    print(f"Phase F2 complete in {TIMINGS['F2']:.1f}s -> {out_csv_F2.name}, "
          f"{out_csv_F2b.name}")

# Phase J — visible-functional protection and certificate scope

Shallow protection is not a kernel property of the full gradient mode:
at parameter (1,0) the mode has $\lambda_{\mathrm{mode}} = 3.87$ with
97% unprotected weight, so it does not lie in the generator kernel. The
correct statement, verified here for every active shallow parameter, is
that the readout-visible pairing carries zero weight on unprotected pairs,
which yields a measured ratio identically 1 at every strength (protected
pairs are exact fixed points of the correlated channel). The phase also
demonstrates the dissipative hypothesis of the certificate: the pure
coherent site-Z control has zero Hermitian rate yet a nonzero measured
response, so zero decay rates certify nothing about coherent noise.


In [ ]:
out_csv_J = OUT_DIR / "phaseJ_visible_protection.csv"
if cache_or_compute(out_csv_J, "Phase J"):
    print("=" * 72)
    print("PHASE J -- VISIBLE-FUNCTIONAL PROTECTION AND CERTIFICATE SCOPE")
    print("=" * 72)
    t0 = time.time(); rows = []
    ch = SUITE["corr_dephase"]
    L_rJ, pairsJ = car.restricted_generator(ch, N, R, GAMMA_PROBE)
    ratesJ = cgg.numeric_diag_rates(L_rJ)
    maskJ = cgg.protected_mask(ratesJ, tol=1e-6)
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    PARAMS_J = [(ell, q) for ell in range(L) for q in range(N)
                if act[ell, q] > 1e-4 * act.max()]
    erng = np.random.default_rng(EST_SEED)
    est_draws = [erng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                 for _ in range(CFG["n_theta_est"])]
    hrng = np.random.default_rng(HESS_SEED)
    m2_draws = [hrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                for _ in range(CFG["n_theta_hess"])]
    coh = cac.SiteZOverRotation()
    for (ell_i, q_i) in PARAMS_J:
        # full-mode decay rate (rate-weighted Rayleigh quotient)
        lam_vals, phis = [], []
        for th in est_draws:
            for ell, M in car.slot_modes(MODEL, th, BETA, ell_i, q_i).items():
                g = car.vec_offdiag(M, pairsJ)
                _, phi, _, lam = cgg.support_cut_bound(g, ratesJ, maskJ)
                if np.isfinite(lam):
                    lam_vals.append(lam); phis.append(phi)
        lam_mode = float(np.mean(lam_vals))
        phi_mode = float(np.mean(phis))
        # readout-visible pairing: fraction of visible weight unprotected
        fr = []
        for th in est_draws:
            w = cgg.visible_pair_weights(MODEL, th, BETA, ell_i, q_i, pairsJ)
            f = cgg.unprotected_fraction(w, maskJ)
            if np.isfinite(f):
                fr.append(f)
        vis_unprot = float(np.mean(fr)) if fr else float("nan")
        # all-orders check of the measured ratio at strong noise
        ratios = {}
        for g in CFG["j_gammas"]:
            vals0, valsg = [], []
            for th in m2_draws:
                g0 = cgg.grad_slotwise(MODEL, th, BETA, None,
                                       np.zeros(L), ell_i, q_i, SHIFT)
                gg = cgg.grad_slotwise(MODEL, th, BETA, ch,
                                       np.full(L, g), ell_i, q_i, SHIFT)
                vals0.append(g0 ** 2); valsg.append(gg ** 2)
            ratios[g] = float(np.sum(valsg) / np.sum(vals0))
        # coherent scope: signed control with zero Hermitian rate
        vals0, valsc = [], []
        for th in m2_draws:
            g0 = cgg.grad_slotwise(MODEL, th, BETA, None,
                                   np.zeros(L), ell_i, q_i, SHIFT)
            gc = cgg.grad_slotwise(MODEL, th, BETA, coh,
                                   np.full(L, 0.02), ell_i, q_i, SHIFT)
            vals0.append(g0 ** 2); valsc.append(gc ** 2)
        coh_delta = float(1.0 - np.sum(valsc) / np.sum(vals0))
        gk = sorted(ratios)
        rows.append({"ell": ell_i, "q": q_i,
                     "lambda_mode": lam_mode,
                     "phi_mode_unprotected": phi_mode,
                     "visible_unprotected_fraction": vis_unprot,
                     f"ratio_gamma_{gk[0]}": ratios[gk[0]],
                     f"ratio_gamma_{gk[1]}": ratios[gk[1]],
                     "all_orders_dev": float(max(abs(ratios[g] - 1.0)
                                                 for g in ratios)),
                     "coherent_delta_tilde_g002": coh_delta})
        print(f"  ({ell_i},{q_i}): lambda_mode {lam_mode:7.4f} "
              f"(phi {phi_mode:.3f})  visible unprot {vis_unprot:.2e}  "
              f"ratio dev {rows[-1]['all_orders_dev']:.2e}  "
              f"coherent delta {coh_delta:+.4f}")
    pd.DataFrame(rows).to_csv(out_csv_J, index=False)
    write_meta(out_csv_J, "J")
    TIMINGS["J"] = time.time() - t0
    print(f"Phase J complete in {TIMINGS['J']:.1f}s -> {out_csv_J.name}")
    print("  Reading: protection is a property of the readout-visible")
    print("  pairing, not of the full gradient mode, and the certificate")
    print("  requires a dissipative channel: the coherent control has zero")
    print("  Hermitian rate yet a nonzero measured response.")

# Phase K — response-weight overlap in the two-excitation sector

Phase G's protected-pair counts are graph statistics, not gradient
degradation. The operative quantity is the overlap of the visible
response weight with the protected subspace, computed here per parameter
at $r = 2$ with the delocalised (spread) input, alongside a CRN-measured
alignment, with the $r = 1$ combinatorially protected case as the
reference.


In [ ]:
out_csv_K = OUT_DIR / "phaseK_r2_overlap.csv"
if cache_or_compute(out_csv_K, "Phase K"):
    print("=" * 72)
    print("PHASE K -- RESPONSE-WEIGHT OVERLAP IN THE TWO-EXCITATION SECTOR")
    print("=" * 72)
    t0 = time.time(); rows = []
    print("  r=2 input: delocalised (spread), the regime in which the"
          " companion study measured the protection collapse")
    R2 = 2
    ch = SUITE["corr_dephase"]
    model2 = cac.Brickwork(N, L, R2, init_state="spread")
    pairs2 = cac.pair_basis(N, R2)
    ana2 = np.array([cgg.analytic_rate_pair(ch, a, b, N)
                     for (a, b) in pairs2])
    mask2 = cgg.protected_mask(ana2, tol=1e-9)
    lam2 = float(ana2.max())
    act2 = cab.activity_preflight(model2, BETA, n_draw=CFG["preflight_draws"],
                                  delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    params2 = cab.pick_parameters(act2, k=2, distinct_layers=True)
    erng = np.random.default_rng(EST_SEED)
    est_draws = [erng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                 for _ in range(CFG["n_theta_est"])]
    mrng = np.random.default_rng(MEAS_SEED)
    meas_draws = [mrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                  for _ in range(CFG["k_meas"])]
    for (ell_i, q_i) in params2:
        fr = []
        for th in est_draws:
            w = cgg.visible_pair_weights(model2, th, BETA, ell_i, q_i, pairs2)
            f = cgg.unprotected_fraction(w, mask2)
            if np.isfinite(f):
                fr.append(f)
        vis_unprot = float(np.mean(fr)) if fr else float("nan")
        g = 0.02
        s0 = sg = 0.0
        for th in meas_draws:
            g0 = cgg.grad_slotwise(model2, th, BETA, None,
                                   np.zeros(L), ell_i, q_i, SHIFT)
            gg = cgg.grad_slotwise(model2, th, BETA, ch,
                                   np.full(L, g), ell_i, q_i, SHIFT)
            s0 += g0 ** 2; sg += gg ** 2
        delta = 1.0 - sg / s0
        om = delta / (2 * g * L * lam2)
        rows.append({"r": R2, "ell": ell_i, "q": q_i,
                     "visible_unprotected_fraction": vis_unprot,
                     "visible_protected_fraction": 1.0 - vis_unprot,
                     "omega_hat_g002": float(om),
                     "protected_pair_fraction_graph": float(mask2.mean())})
        print(f"  r=2 ({ell_i},{q_i}): visible protected "
              f"{1-vis_unprot:.3f}  measured omega {om:.3f}  "
              f"(graph protected fraction {mask2.mean():.3f})")
    # r=1 comparison at a combinatorially protected parameter
    L_r1, pairs1 = car.restricted_generator(ch, N, 1, GAMMA_PROBE)
    mask1 = cgg.protected_mask(cgg.numeric_diag_rates(L_r1), tol=1e-6)
    fr1 = []
    for th in est_draws:
        w = cgg.visible_pair_weights(MODEL, th, BETA, 2, 1, pairs1)
        f = cgg.unprotected_fraction(w, mask1)
        if np.isfinite(f):
            fr1.append(f)
    rows.append({"r": 1, "ell": 2, "q": 1,
                 "visible_unprotected_fraction": float(np.mean(fr1)),
                 "visible_protected_fraction": 1.0 - float(np.mean(fr1)),
                 "omega_hat_g002": 0.0,
                 "protected_pair_fraction_graph": float(mask1.mean())})
    pd.DataFrame(rows).to_csv(out_csv_K, index=False)
    write_meta(out_csv_K, "K")
    TIMINGS["K"] = time.time() - t0
    print(f"Phase K complete in {TIMINGS['K']:.1f}s -> {out_csv_K.name}")
    print("  Reading: the graph counts alone do not measure trainability;")
    print("  the overlap of the visible response weight with the protected")
    print("  subspace is the operative quantity, and it collapses at r=2.")

# Phase F2D — slot exponents derived first, then tested

The exponents are derived from the channel action on the visible support
(every visible $r=1$ coherence differs at exactly two sites; the
final-slot pairing identity removes the last dephasing slot), recorded in
the source, and the measured product law is then tested against the
derived values on random per-slot strengths.


In [ ]:
out_csv_F2D = OUT_DIR / "phaseF2_derived.csv"
if cache_or_compute(out_csv_F2D, "Phase F2D"):
    print("=" * 72)
    print("PHASE F2D -- SLOT EXPONENTS DERIVED FIRST, THEN TESTED")
    print("=" * 72)
    t0 = time.time()
    # Derivation (recorded before measurement, correcting the v1.2 logic
    # which inferred exponents from the data): every visible r = 1
    # coherence differs at exactly two sites, so a single isotropic
    # dephasing slot multiplies each visible amplitude by (1-2g)^2 and a
    # damping slot by (1-g); squaring for the second moment gives the
    # per-slot exponent 2 r_ell / alpha with r_ell the analytic
    # slot-resolved response rate (4 or 0 for dephasing at L = 3 by the
    # final-slot pairing identity; 1 on every slot for damping), i.e.
    #   dephasing: (4, 4, 0)   damping: (2, 2, 2)   correlated: (0, 0, 0).
    DERIVED = {"dephase": [4, 4, 0], "amp_damp": [2, 2, 2],
               "corr_dephase": [0, 0, 0]}
    ALPHA = {"dephase": 2.0, "amp_damp": 1.0, "corr_dephase": 2.0}
    hrng = np.random.default_rng(HESS_SEED)
    theta_draws = [hrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                   for _ in range(CFG["n_theta_hess"])]
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    (ELL_F, Q_F) = cab.pick_parameters(act, k=1)[0]
    prng = np.random.default_rng(SWEEP_SEED + 11)
    rows = []
    for name, ks in DERIVED.items():
        ch = SUITE[name]; alpha = ALPHA[name]
        def M2(gvec):
            return cgg.m2_slotwise(MODEL, BETA, ch, gvec, ELL_F, Q_F,
                                   theta_draws, SHIFT)
        m0 = M2(np.zeros(L))
        dev = 0.0
        for _ in range(CFG["f2_product_draws"]):
            gv = prng.uniform(0.0, 0.25, size=L)
            pred = float(np.prod([(1 - alpha * gv[l]) ** ks[l]
                                  for l in range(L)]))
            dev = max(dev, abs(M2(gv) / m0 - pred))
        rows.append({"channel": name, "ell": ELL_F, "q": Q_F,
                     "derived_exponents": str(ks),
                     "alpha": alpha,
                     "Lambda1_derived": alpha * sum(ks) / 2.0,
                     "K2_derived": alpha**2 * sum(ks) * (sum(ks) - 1) / 2.0,
                     "law_maxdev_vs_derived": dev})
        print(f"  {name:14s} derived {ks}: measured law deviates by "
              f"{dev:.2e}")
    pd.DataFrame(rows).to_csv(out_csv_F2D, index=False)
    write_meta(out_csv_F2D, "F2D")
    TIMINGS["F2D"] = time.time() - t0
    print(f"Phase F2D complete in {TIMINGS['F2D']:.1f}s -> "
          f"{out_csv_F2D.name}")

# Phase J2 — exact fixed-point witness and theorem hypotheses

The all-orders protection statement is a theorem with hypotheses, not a
two-strength observation: protected pairs are exact fixed points of the
correlated channel at every strength (each edge factor equals one when
parities agree), and slot-wise visible confinement then leaves the
measured gradient unchanged. The hypotheses are a dissipative channel of
the Z-diagonal family and confinement at every slot; Phase J's coherent
control shows the dissipative hypothesis is not removable. This phase
computes the exact fixed-point witness and the minimum change on
unprotected pairs.


In [ ]:
out_csv_J2 = OUT_DIR / "phaseJ2_fixed_points.csv"
if cache_or_compute(out_csv_J2, "Phase J2"):
    print("=" * 72)
    print("PHASE J2 -- EXACT FIXED-POINT WITNESS FOR THE ALL-ORDERS THEOREM")
    print("=" * 72)
    t0 = time.time()
    # Theorem structure (stated with its hypotheses): (i) every protected
    # pair is an EXACT fixed point of the correlated channel at every
    # strength, since each edge factor is (1-g) + g p_e(a) p_e(b) = 1 when
    # the parities agree; (ii) if the visible pairing at every slot is
    # supported on protected pairs, the measured gradient is unchanged at
    # every strength. Hypotheses: dissipative pair-diagonal channel of the
    # Z-diagonal family; the coherent site-Z control (zero Hermitian rate,
    # nonzero response, Phase J) shows the dissipative hypothesis is not
    # removable. This phase computes the exact witness for (i).
    ch = SUITE["corr_dephase"]
    L_rw, pairsW = car.restricted_generator(ch, N, R, GAMMA_PROBE)
    maskW = cgg.protected_mask(cgg.numeric_diag_rates(L_rw), tol=1e-6)
    rows = []
    for g in (0.1, 0.3, 0.5):
        worst_prot, worst_unprot_min = 0.0, np.inf
        for m, (a, b) in zip(maskW, pairsW):
            E = np.zeros((2**N, 2**N), dtype=complex)
            E[a, b] = 1.0
            out = ch.apply(E, g, N)
            dev = float(np.max(np.abs(out - E)))
            if m:
                worst_prot = max(worst_prot, dev)
            else:
                worst_unprot_min = min(worst_unprot_min, dev)
        rows.append({"gamma": g,
                     "protected_max_dev_from_identity": worst_prot,
                     "unprotected_min_change": float(worst_unprot_min)})
        print(f"  gamma={g:.1f}: protected pairs fixed to {worst_prot:.1e}"
              f"; every unprotected pair changes by >= "
              f"{worst_unprot_min:.3f}")
    pd.DataFrame(rows).to_csv(out_csv_J2, index=False)
    write_meta(out_csv_J2, "J2")
    TIMINGS["J2"] = time.time() - t0
    print(f"Phase J2 complete in {TIMINGS['J2']:.1f}s -> {out_csv_J2.name}")

# Phase K2 — sector and input state separated (2x2 factorial)

The seeded spread input already breaks protection within $r = 1$
(support-weight protected fraction 0.83, degradation 4.5% at the
combinatorially protected parameter), so a comparison across sectors
alone would confound sector, input and parameter. This factorial
separates them; the localised $r = 2$ cell has no readout-visible
parameters at this geometry and is recorded as such. The support-weight
diagnostic (absolute-magnitude sums) is a support measure, not a signed
susceptibility, and measured degradation is reported alongside it. The
supported conclusion: in the audited $r=1$ control, changing the input
alone is sufficient to break perfect protection; a sector main effect is
neither isolated nor excluded.


In [ ]:
out_csv_K2 = OUT_DIR / "phaseK2_factorial.csv"
if cache_or_compute(out_csv_K2, "Phase K2"):
    print("=" * 72)
    print("PHASE K2 -- SECTOR AND INPUT STATE SEPARATED (2x2 FACTORIAL)")
    print("=" * 72)
    t0 = time.time()
    # v1.3 correction: Phase K changed sector, input state and parameter
    # together, so it could not attribute the loss of protection to the
    # sector. The audit's control showed the spread input already breaks
    # protection within r = 1. This phase runs the full factorial. The
    # localised r = 2 cell has no readout-visible parameters at this
    # geometry (preflight at the numerical floor), which is recorded as a
    # finding rather than silently skipped. The support-weight diagnostic
    # sums absolute magnitudes and is a support measure, not a signed
    # susceptibility; measured degradation is reported alongside it.
    ch = SUITE["corr_dephase"]
    erng = np.random.default_rng(EST_SEED)
    est_draws = [erng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                 for _ in range(CFG["n_theta_est"])]
    mrng = np.random.default_rng(MEAS_SEED)
    meas_draws = [mrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                  for _ in range(CFG["k_meas"])]
    rows = []
    for R_s, init, note in ((1, "localized", "reference"),
                            (1, "spread", "input control"),
                            (2, "localized", "no visible parameters"),
                            (2, "spread", "delocalised higher sector")):
        try:
            model_s = cac.Brickwork(N, L, R_s, init_state=init)
        except TypeError:
            model_s = cac.Brickwork(N, L, R_s)
        pairsS = cac.pair_basis(N, R_s)
        anaS = np.array([cgg.analytic_rate_pair(ch, a, b, N)
                         for (a, b) in pairsS])
        maskS = cgg.protected_mask(anaS, tol=1e-9)
        lamS = float(anaS.max())
        act = cab.activity_preflight(model_s, BETA,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        if act.max() < 1e-10:
            rows.append({"sector_r": R_s, "input": init, "ell": -1, "q": -1,
                         "support_weight_protected_fraction": float("nan"),
                         "delta_tilde_g002": float("nan"),
                         "omega_hat_g002": float("nan"), "note": note})
            print(f"  r={R_s} {init:9s}: no readout-visible parameters "
                  f"(max activity {act.max():.1e}) -- recorded")
            continue
        if R_s == 1:
            params = [(2, 1)]           # the combinatorially protected case
        else:
            params = cab.pick_parameters(act, k=2, distinct_layers=True)
        for (ell_i, q_i) in params:
            fr = []
            for th in est_draws:
                w = cgg.visible_pair_weights(model_s, th, BETA,
                                             ell_i, q_i, pairsS)
                f = cgg.unprotected_fraction(w, maskS)
                if np.isfinite(f):
                    fr.append(f)
            vis_unprot = float(np.mean(fr)) if fr else float("nan")
            g = 0.02
            s0 = sg = 0.0
            for th in meas_draws:
                g0 = cgg.grad_slotwise(model_s, th, BETA, None,
                                       np.zeros(L), ell_i, q_i, SHIFT)
                gg = cgg.grad_slotwise(model_s, th, BETA, ch,
                                       np.full(L, g), ell_i, q_i, SHIFT)
                s0 += g0 ** 2; sg += gg ** 2
            delta = float(1.0 - sg / s0)
            rows.append({"sector_r": R_s, "input": init,
                         "ell": ell_i, "q": q_i,
                         "support_weight_protected_fraction":
                             1.0 - vis_unprot,
                         "delta_tilde_g002": delta,
                         "omega_hat_g002": delta / (2 * g * L * lamS),
                         "note": note})
            print(f"  r={R_s} {init:9s} ({ell_i},{q_i}): support-weight "
                  f"protected {1-vis_unprot:.4f}  degradation "
                  f"{delta:+.4f}")
    pd.DataFrame(rows).to_csv(out_csv_K2, index=False)
    write_meta(out_csv_K2, "K2")
    TIMINGS["K2"] = time.time() - t0
    print(f"Phase K2 complete in {TIMINGS['K2']:.1f}s -> {out_csv_K2.name}")
    print("  Reading: delocalisation alone breaks protection within r = 1;")
    print("  the defensible statement concerns delocalised inputs, not the")
    print("  sector number as such.")

# Phase H2 — difference-based equivalence and replication

Overlap of two marginal intervals is not an equivalence test, since wide
intervals produce apparent agreement precisely when the experiment is
uninformative, and a single ensemble cannot separate estimator bias from
sampling fluctuation. This phase therefore decides the depth-six question
on the **difference** between the extrapolated measurement and the
prediction, both bootstrapped at draw level: **equivalent** if the 90%
difference interval lies inside the $\pm 0.01$ margin, **different** if
the 95% interval excludes the margin entirely, **inconclusive** otherwise,
with depth five as the control. A matched-draw check computes the
first-order object directly on the measurement draw set, isolating
implementation error from ensemble variation, and a replicated
ensemble-size study at two estimator sizes decides between bias and
sampling fluctuation before any bias language is used.


In [ ]:
out_csv_H2 = OUT_DIR / "phaseH2_equivalence.csv"
out_csv_H2b = OUT_DIR / "phaseH2_replication.csv"
if cache_or_compute(out_csv_H2, "Phase H2"):
    print("=" * 72)
    print("PHASE H2 -- DIFFERENCE-BASED EQUIVALENCE AND REPLICATION")
    print("=" * 72)
    t0 = time.time()
    ch = SUITE["corr_dephase"]
    L_rH, _ = car.restricted_generator(ch, N, R, GAMMA_PROBE)
    lam_H = float(cgg.hermitian_rate_spectrum(L_rH)[-1])
    MARGIN = CFG["eq_margin"]
    gs = np.array(CFG["gamma_stress"], dtype=float)
    B = CFG["bootstrap_B"]
    rows, rep_rows = [], []

    def draw_thetas(seed, n, L_d):
        rng = np.random.default_rng(seed)
        return [rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L_d, N))
                for _ in range(n)]

    def pred_arrays(model_d, beta_d, thetas, ell_i, q_i, L_d):
        """Per-draw response rate sums and df0^2 weights."""
        r_list, w_list = [], []
        for th in thetas:
            rr = car.response_rates(model_d, th, beta_d, ch, ell_i, q_i,
                                    GAMMA_PROBE)
            r_list.append(float(sum(rr["rates"].values())))
            w_list.append(float(rr["df0"]) ** 2)
        return np.array(r_list), np.array(w_list)

    def meas_arrays(model_d, beta_d, thetas, ell_i, q_i, L_d):
        """CRN gradients: noiseless squared and noisy squared per strength."""
        g0 = np.array([cgg.grad_slotwise(model_d, th, beta_d, None,
                                         np.zeros(L_d), ell_i, q_i,
                                         SHIFT) ** 2 for th in thetas])
        gn = {}
        for g in gs:
            gn[float(g)] = np.array(
                [cgg.grad_slotwise(model_d, th, beta_d, ch,
                                   np.full(L_d, float(g)), ell_i, q_i,
                                   SHIFT) ** 2 for th in thetas])
        return g0, gn

    def intercept(g0s, gns, L_d):
        oms = [(1.0 - float(gns[float(g)].sum()) / float(g0s.sum()))
               / (2 * g * L_d * lam_H) for g in gs]
        return float(np.polyval(np.polyfit(gs, oms, 2), 0.0))

    for L_d in CFG["stress_depths"]:
        role = "stress" if L_d == max(CFG["stress_depths"]) else "control"
        model_d = cac.Brickwork(N, L_d, R)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        act = cab.activity_preflight(model_d, beta_d,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        params = cab.pick_parameters(act, k=2, distinct_layers=True)
        for (ell_i, q_i) in params:
            est_th = draw_thetas(EST_SEED, CFG["stress_est"], L_d)
            r_arr, w_arr = pred_arrays(model_d, beta_d, est_th,
                                       ell_i, q_i, L_d)
            pred = float(np.sum(w_arr * r_arr) / np.sum(w_arr)) \
                / (L_d * lam_H)
            meas_th = draw_thetas(MEAS_SEED, CFG["stress_meas"], L_d)
            g0s, gns = meas_arrays(model_d, beta_d, meas_th,
                                   ell_i, q_i, L_d)
            om0 = intercept(g0s, gns, L_d)
            # matched-draw implementation check: the same first-order
            # object computed directly on the measurement draw set
            r_m, w_m = pred_arrays(model_d, beta_d, meas_th,
                                   ell_i, q_i, L_d)
            pred_on_meas = float(np.sum(w_m * r_m) / np.sum(w_m)) \
                / (L_d * lam_H)
            # joint bootstrap of the DIFFERENCE (independent ensembles)
            brng = np.random.default_rng(BOOT_SEED + 200)
            nE, nM = len(r_arr), len(g0s)
            diffs = []
            for _ in range(B):
                ie = brng.integers(0, nE, size=nE)
                pb = float(np.sum(w_arr[ie] * r_arr[ie])
                           / np.sum(w_arr[ie])) / (L_d * lam_H)
                im = brng.integers(0, nM, size=nM)
                g0b = g0s[im]
                gnb = {k: v[im] for k, v in gns.items()}
                mb = intercept(g0b, gnb, L_d)
                diffs.append(mb - pb)
            diffs = np.array(diffs)
            d50 = float(np.median(diffs))
            d90 = np.percentile(diffs, [5, 95])
            d95 = np.percentile(diffs, [2.5, 97.5])
            if d90[0] >= -MARGIN and d90[1] <= MARGIN:
                verdict = "equivalent"
            elif d95[0] > MARGIN or d95[1] < -MARGIN:
                verdict = "different"
            else:
                verdict = "inconclusive"
            rows.append({"depth": L_d, "ell": ell_i, "q": q_i, "role": role,
                         "omega_pred": pred, "om_extrap": om0,
                         "pred_on_meas_draws": pred_on_meas,
                         "matched_draw_dev": abs(pred_on_meas - om0),
                         "diff_median": d50,
                         "diff_ci90_lo": float(d90[0]),
                         "diff_ci90_hi": float(d90[1]),
                         "diff_ci95_lo": float(d95[0]),
                         "diff_ci95_hi": float(d95[1]),
                         "margin": MARGIN, "verdict": verdict})
            print(f"  L={L_d} ({ell_i},{q_i}) [{role:7s}] pred {pred:.4f} "
                  f"extrap {om0:.4f}  diff90 [{d90[0]:+.4f},{d90[1]:+.4f}]"
                  f"  matched dev {abs(pred_on_meas-om0):.1e} -> {verdict}")
            # replicated ensemble-size study (stress depth only)
            if role == "stress":
                for size in CFG["h2_sizes"]:
                    for rep in range(CFG["h2_reps"]):
                        th = draw_thetas(EST_SEED + 1000 * (rep + 1),
                                         size, L_d)
                        ra, wa = pred_arrays(model_d, beta_d, th,
                                             ell_i, q_i, L_d)
                        pv = float(np.sum(wa * ra) / np.sum(wa)) \
                            / (L_d * lam_H)
                        rep_rows.append({"depth": L_d, "ell": ell_i,
                                         "q": q_i, "size": size,
                                         "rep": rep, "omega_pred": pv})
    pd.DataFrame(rows).to_csv(out_csv_H2, index=False)
    rp = pd.DataFrame(rep_rows)
    rp.to_csv(out_csv_H2b, index=False)
    write_meta(out_csv_H2b, "H2")
    write_meta(out_csv_H2, "H2", companions=[out_csv_H2b.name])
    if len(rp):
        summ = rp.groupby(["ell", "q", "size"]).omega_pred \
                 .agg(["mean", "std"]).reset_index()
        print("  replication (stress depth): omega_pred by estimator size")
        for _, r in summ.iterrows():
            print(f"    ({int(r['ell'])},{int(r['q'])}) size "
                  f"{int(r['size']):3d}: {r['mean']:.4f} "
                  f"+/- {r['std']:.4f}")
        for (e_, q_), g in summ.groupby(["ell", "q"]):
            g = g.sort_values("size")
            if len(g) == 2:
                gap = abs(g["mean"].iloc[0] - g["mean"].iloc[1])
                spread = float(np.hypot(g["std"].iloc[0], g["std"].iloc[1]))
                tag = ("bias-consistent" if gap > spread
                       else "within sampling fluctuation")
                print(f"    ({e_},{q_}): size means differ by {gap:.4f} "
                      f"vs combined spread {spread:.4f} -> {tag}")
    TIMINGS["H2"] = time.time() - t0
    print(f"Phase H2 complete in {TIMINGS['H2']:.1f}s -> {out_csv_H2.name}, "
          f"{out_csv_H2b.name}")

# Phase J3 — path-support certificate

Per-slot ideal-pairing protection does not imply protection under joint
noise insertions: a two-slot circuit with $g(\gamma_1,\gamma_2)
= (1-\gamma_1\gamma_2)/\sqrt{2}$ whose ideal per-slot visible pairing is
entirely protected makes the failure explicit. The correct sufficient
condition quantifies over **paths**: every matrix-unit path with nonzero
transfer amplitude from the (shifted) input to the readout must lie in the
channel's exact fixed-point set at every noise slot. Then every slot acts
as the identity on every contributing path at every strength, jointly, so
the measured gradient is exactly invariant; the proof combines Phase J2's
fixed-point witness with induction over slots, under the dissipative
Z-diagonal family hypothesis. This phase computes the certificate in its
purely combinatorial form, the transfer relation built from the permitted
hopping edges alone and fixed pairs from the analytic parity rule, with no
linear algebra, amplitude thresholds or probes, so it is conservative for
every gate value; the numerical-support version is retained as a sharper
cross-check and asserted to be contained in the combinatorial sets. The
construction is parameter-independent (a derivative insertion never
enlarges support), making it a conservative whole-readout certificate for
the chosen input, subsequently checked on each active gradient; it is not
a parameter-selective path analysis. The reference-geometry verdicts and
slot pair sets (slots 0 and 1 carry only the protected pair and its
transpose beyond diagonals; slot 2 diagonals only, matching an independent
path enumeration) are positively asserted, alongside the joint
all-strength invariance the certificate implies.


In [ ]:
out_csv_J3 = OUT_DIR / "phaseJ3_path_certificate.csv"
if cache_or_compute(out_csv_J3, "Phase J3"):
    print("=" * 72)
    print("PHASE J3 -- PATH-SUPPORT CERTIFICATE (CORRECTED THEOREM)")
    print("=" * 72)
    t0 = time.time()
    # The audit's counterexample (a two-slot circuit whose ideal per-slot
    # visible pairing is fully protected yet whose gradient changes at
    # mixed second order, g = (1 - g1 g2)/sqrt(2)) shows that per-slot
    # ideal-pairing protection does NOT imply protection under joint
    # insertions. The corrected sufficient condition is PATH SUPPORT:
    # every matrix-unit path with nonzero transfer amplitude from the
    # (shifted) input to the readout must lie in the channel's exact
    # fixed-point set at every noise slot. Then each slot acts as the
    # identity on every contributing path at every strength, jointly, and
    # the measured gradient is exactly noise-invariant. This phase
    # computes that certificate by boolean pair-level reachability in the
    # sector (no cancellation assumptions), checks it against the audit's
    # path enumeration for the shallow geometry, and verifies the joint
    # multi-slot invariance it implies.
    ch = SUITE["corr_dephase"]
    SEC = cac.sector_basis(N, R)
    dS = len(SEC)
    idxS = {s: k for k, s in enumerate(SEC)}
    # PRIMARY certificate inputs, purely combinatorial (audit round 4):
    # the transfer relation comes from the permitted hopping edges alone
    # (a site can stay or move across an active edge), and the fixed
    # pairs come from the analytic parity rule (a pair is fixed iff every
    # correlated edge sees equal parities), with no linear algebra, no
    # amplitude thresholds and no probe. This is conservative for every
    # gate value. The numerical-support version (thresholded unitaries,
    # numeric rates) is retained as a sharper cross-check and must be
    # contained in the combinatorial sets.
    fixed_pairs = {(a, b) for a in SEC for b in SEC
                   if cgg.analytic_rate_pair(ch, a, b, N) == 0.0}
    def adjacency_mask(ell):
        A = np.eye(dS, dtype=bool)
        for j in MODEL.E[ell]:
            u, v = j, (j + 1) % N
            iu, iv = idxS[1 << u], idxS[1 << v]
            A[iu, iv] = A[iv, iu] = True
        return A
    TOL_AMP = 1e-12
    def transfer_mask(U):
        return np.abs(U[np.ix_(SEC, SEC)]) > TOL_AMP

    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    PARAMS_J3 = [(ell, q) for ell in range(L) for q in range(N)
                 if act[ell, q] > 1e-4 * act.max()]
    hrng = np.random.default_rng(HESS_SEED)
    m2_draws = [hrng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                for _ in range(CFG["n_theta_hess"])]
    def slot_sets(mask_fn):
        """Contributing pair sets per slot under a given per-layer
        transfer relation. The construction is parameter-independent (the
        derivative insertion never enlarges support), so it is a
        conservative WHOLE-READOUT certificate for the chosen input,
        checked afterwards on each active gradient; it is not a
        parameter-selective path analysis."""
        fwd = np.zeros((dS, dS), dtype=bool)
        fwd[0, 0] = True          # localised input |0><0| (sector index 0)
        fwd_slots = []
        for ell in range(L):
            A = mask_fn(ell)
            fwd = (A @ fwd @ A.T) > 0
            fwd_slots.append(fwd.copy())
        bwd = np.eye(dS, dtype=bool)
        bwd_slots = [None] * L
        bwd_slots[L - 1] = bwd.copy()
        for ell in range(L - 2, -1, -1):
            A = mask_fn(ell + 1)
            bwd = (A.T @ bwd @ A) > 0
            bwd_slots[ell] = bwd.copy()
        out = {}
        for ell in range(L):
            contrib = fwd_slots[ell] & bwd_slots[ell]
            out[ell] = {(i, j) for i in range(dS) for j in range(dS)
                        if contrib[i, j]}
        return out

    comb_sets = slot_sets(adjacency_mask)
    num_sets = slot_sets(
        lambda ell: transfer_mask(MODEL.xy_layer(BETA[ell], ell)))
    comb_ok = all((SEC[i], SEC[j]) in fixed_pairs
                  for ell in comb_sets for (i, j) in comb_sets[ell])
    for ell in range(L):
        assert num_sets[ell] <= comb_sets[ell],             "numerical support escaped the combinatorial relation"
    # reference-case positive assertions (audit round 4): at the shallow
    # n = 8 geometry the certificate must HOLD, with slots 0 and 1
    # carrying only the protected pair (sites 0,1) and its transpose
    # beyond diagonals, and slot 2 diagonals only
    if N == 8 and L == 3:
        assert comb_ok, "reference-case combinatorial certificate failed"
        i01, i10 = (idxS[1], idxS[2]), (idxS[2], idxS[1])
        for ell in (0, 1):
            off = {p for p in comb_sets[ell] if p[0] != p[1]}
            assert off <= {i01, i10}, f"unexpected slot-{ell} pairs {off}"
        assert all(i == j for (i, j) in comb_sets[2]),             "slot 2 must be diagonal only"
    rows = []
    for (ell_i, q_i) in PARAMS_J3:
        cert_ok = comb_ok
        slot_pairs = {ell: sorted(comb_sets[ell]) for ell in range(L)}
        # the invariance the certificate implies: joint all-slot noise at
        # strong strength leaves every gradient draw exactly unchanged
        worst = 0.0
        for th in m2_draws:
            g0 = cgg.grad_slotwise(MODEL, th, BETA, None,
                                   np.zeros(L), ell_i, q_i, SHIFT)
            gj = cgg.grad_slotwise(MODEL, th, BETA, ch,
                                   np.full(L, 0.3), ell_i, q_i, SHIFT)
            worst = max(worst, abs(gj - g0))
        rows.append({"ell": ell_i, "q": q_i,
                     "path_certificate": bool(cert_ok),
                     "certificate_kind": "combinatorial "
                     "(edge relation + analytic parity)",
                     "slot_pair_sets": str({k: v for k, v in
                                            slot_pairs.items()}),
                     "joint_invariance_maxdev": worst})
        print(f"  ({ell_i},{q_i}): path certificate "
              f"{'HOLDS' if cert_ok else 'fails'}; joint all-slot "
              f"invariance dev {worst:.2e}")
        assert (not cert_ok) or worst < 1e-10, \
            "certificate held but joint invariance failed"
    pd.DataFrame(rows).to_csv(out_csv_J3, index=False)
    write_meta(out_csv_J3, "J3")
    TIMINGS["J3"] = time.time() - t0
    print(f"Phase J3 complete in {TIMINGS['J3']:.1f}s -> {out_csv_J3.name}")
    print("  Reading: the PRIMARY certificate is purely combinatorial")
    print("  (edge relation + analytic parity, no linear algebra); the")
    print("  numerical-support sets are verified to be contained in it.")
    print("  The certificate quantifies over PATHS, so it")
    print("  survives simultaneous noise insertions, which the per-slot")
    print("  ideal-pairing condition (audit counterexample) does not.")

# Phase P — deterministic population benchmark (prior-integrated)

The population second moment over the uniform small-box prior is computed
exactly in the eight-dimensional single-excitation sector by two-copy
tensor propagation, each phase layer averaged in closed form via
$\mathbb{E}[e^{ik\theta}] = \sin(k\delta)/(k\delta)$ and the
parameter-shift offsets entering as exact constants; the correlated slot
factors are exact at every strength. No Monte Carlo enters.
Cross-validation targets from an independent implementation of the same
quantities are recorded in the source before the run. The phase then
re-expresses the depth-six statistics against this population target:
replicate means at both estimator sizes with their distance from the
target, the paired analysis of the nested size comparison, and the
deterministic extrapolation offset that bounds the probe-and-curvature
error separately from sampling variation.


In [ ]:
out_csv_P = OUT_DIR / "phaseP_benchmark.csv"
out_csv_P2 = OUT_DIR / "phaseP_estimator_stats.csv"
if cache_or_compute(out_csv_P, "Phase P"):
    print("=" * 72)
    print("PHASE P -- DETERMINISTIC POPULATION BENCHMARK (PRIOR-INTEGRATED)")
    print("=" * 72)
    t0 = time.time()
    # Independent implementation of the audit's deterministic benchmark;
    # circuit-dependent two-copy tensor propagation with exact prior
    # averaging (no Monte Carlo; NOT simulation-free). Three quantities
    # are kept separate throughout (audit round 3):
    #   omega_population                       the response predictor's
    #                                          population value (probe
    #                                          factors cancel for this
    #                                          channel)
    #   zero_strength_derivative_over_probe_rate   the true measurement
    #                                          derivative at zero divided
    #                                          by the finite-probe rate
    #   extrapolation_over_probe_rate          the three-strength
    #                                          intercept on the same scale
    # Cross-validation targets (independent audit implementation):
    #   L=5: 0.4000000000 (both parameters)
    #   L=6 (2,0): 0.3709390604   L=6 (3,0): 0.3053608530
    AUDIT_TARGETS = {(6, 2, 0): 0.3709390604, (6, 3, 0): 0.3053608530}
    ch = SUITE["corr_dephase"]
    SEC = cac.sector_basis(N, 1)
    d = len(SEC)
    exc = [int(np.log2(s)) for s in SEC]
    O_diag = np.array([-1.0 if exc[i] == 0 else 1.0 for i in range(d)])
    LAM = 4.0

    def sinc_u(k):
        return 1.0 if k == 0 else float(np.sin(k * DELTA_INIT)
                                        / (k * DELTA_INIT))

    edges = ch.edges
    ndiff = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            ndiff[i, j] = sum(1 for (u, v) in edges
                              if (exc[i] in (u, v)) != (exc[j] in (u, v)))

    def c_factor(gamma):
        return (1.0 - 2.0 * gamma) ** ndiff

    def z_masks(L_d, ell_i, q_i):
        m = np.zeros((N, d))
        for i in range(d):
            m[exc[i], i] = 1.0
        masks = []
        for ell in range(L_d):
            M4 = np.ones((d, d, d, d))
            for q in range(N):
                u1 = m[q][:, None] - m[q][None, :]
                u2 = u1.copy()
                k = u1[:, :, None, None] + u2[None, None, :, :]
                S = np.vectorize(sinc_u)(k)
                if ell == ell_i and q == q_i:
                    F = -(u1[:, :, None, None] * u2[None, None, :, :]) * S
                else:
                    F = S
                M4 = M4 * F
            masks.append(M4)
        return masks

    def population(L_d, ell_i, q_i, gamma, want_tangent):
        model_d = cac.Brickwork(N, L_d, 1)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        Us = [model_d.xy_layer(beta_d[ell], ell)[np.ix_(SEC, SEC)]
              for ell in range(L_d)]
        masks = z_masks(L_d, ell_i, q_i)
        T = np.zeros((d, d, d, d), dtype=complex)
        T[0, 0, 0, 0] = 1.0
        dT = np.zeros_like(T)
        cg = c_factor(gamma)
        cp = -2.0 * ndiff
        for ell in range(L_d):
            T = T * masks[ell]
            dT = dT * masks[ell]
            U = Us[ell]
            for A, conj, ax in ((U, False, 0), (U, True, 1),
                                (U, False, 2), (U, True, 3)):
                M = A.conj() if conj else A
                T = np.moveaxis(np.tensordot(M, np.moveaxis(T, ax, 0),
                                             axes=(1, 0)), 0, ax)
                dT = np.moveaxis(np.tensordot(M, np.moveaxis(dT, ax, 0),
                                              axes=(1, 0)), 0, ax)
            slot1 = cg[:, :, None, None] * np.ones((1, 1, d, d))
            slot2 = cg[None, None, :, :] * np.ones((d, d, 1, 1))
            if want_tangent:
                dT = dT * slot1 * slot2 + \
                    (cp[:, :, None, None] + cp[None, None, :, :]) * T
            T = T * slot1 * slot2
        m2 = float(np.real(np.einsum("aacc,a,c->", T, O_diag, O_diag)))
        dm2 = float(np.real(np.einsum("aacc,a,c->", dT, O_diag, O_diag))) \
            if want_tangent else float("nan")
        return m2, dm2

    rows = []
    for L_d in CFG["stress_depths"]:
        model_d = cac.Brickwork(N, L_d, 1)
        beta_d = cac.teacher_background(TEACHER_SEED, L_d, N)
        act = cab.activity_preflight(model_d, beta_d,
                                     n_draw=CFG["preflight_draws"],
                                     delta_init=DELTA_INIT,
                                     seed=PREFLIGHT_SEED)
        for (ell_i, q_i) in cab.pick_parameters(act, k=2,
                                                distinct_layers=True):
            m2_0, dm2_0 = population(L_d, ell_i, q_i, 0.0, True)
            om_pop = -dm2_0 / (2 * L_d * LAM * m2_0)
            oms = []
            for g in CFG["gamma_stress"]:
                m2_g, _ = population(L_d, ell_i, q_i, float(g), False)
                oms.append((1.0 - m2_g / m2_0) / (2 * g * L_d * LAM))
            gs = np.array(CFG["gamma_stress"], dtype=float)
            om_ext = float(np.polyval(np.polyfit(gs, oms, 2), 0.0))
            probe = 1.0 - GAMMA_PROBE
            rows.append({"depth": L_d, "ell": ell_i, "q": q_i,
                         "omega_population": om_pop,
                         "population_extrapolated_analytic_norm": om_ext,
                         "extrapolation_offset": om_ext - om_pop,
                         "zero_strength_derivative_over_probe_rate":
                             om_pop / probe,
                         "extrapolation_over_probe_rate": om_ext / probe,
                         "M2_0": m2_0})
            print(f"  L={L_d} ({ell_i},{q_i}): omega_pop {om_pop:.10f}  "
                  f"extrap/probe-rate {om_ext/probe:.10f}")
            # deterministic assertions against the recorded targets
            if L_d == 5:
                assert abs(om_pop - 0.4) < 1e-8, "L=5 target failed"
            key = (L_d, ell_i, q_i)
            if key in AUDIT_TARGETS:
                assert abs(om_pop - AUDIT_TARGETS[key]) < 1e-8, \
                    f"audit cross-validation target failed at {key}"
                print(f"    cross-validated against the audit target "
                      f"{AUDIT_TARGETS[key]:.10f}")
    dfP = pd.DataFrame(rows)
    dfP.to_csv(out_csv_P, index=False)

    write_meta(out_csv_P, "P")
    TIMINGS["P"] = time.time() - t0
    print(f"Phase P complete in {TIMINGS['P']:.1f}s -> {out_csv_P.name}")

# ---- estimator statistics: a SEPARATE cache bound to its H2 inputs -----
# (v1.5 audit correction: an output hash proves a file unchanged, not
# that it remains valid for the current inputs; the cheap statistics are
# cached apart from the expensive integration so an H2 change refreshes
# only the summary)
rep_p = OUT_DIR / "phaseH2_replication.csv"
h2_p = OUT_DIR / "phaseH2_equivalence.csv"
if cache_or_compute(out_csv_P2, "P-stats", inputs=[rep_p, h2_p]):
    t0s = time.time()
    dfP = pd.read_csv(out_csv_P)
    try:
        from scipy.stats import t as _t_dist
        def tcrit(df_):
            return float(_t_dist.ppf(0.975, df_))
    except ImportError:
        T_CRIT = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571,
                  6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228,
                  14: 2.145, 19: 2.093, 29: 2.045}
        def tcrit(df_):
            if df_ not in T_CRIT:
                raise ValueError(
                    f"no tabulated t quantile for df={df_} and scipy is "
                    "unavailable; refusing a silent normal approximation")
            return T_CRIT[df_]
    stat_rows = []
    if rep_p.exists() and h2_p.exists():
        rp = pd.read_csv(rep_p)
        h2 = pd.read_csv(h2_p)
        for _, r in dfP[dfP.depth == max(CFG["stress_depths"])].iterrows():
            sub = rp[(rp.ell == r.ell) & (rp.q == r.q)]
            pop = float(r.omega_population)
            for size, g in sub.groupby("size"):
                m, sd, nrep = (g.omega_pred.mean(), g.omega_pred.std(),
                               len(g))
                sem = sd / np.sqrt(nrep)
                tc = tcrit(nrep - 1)
                stat_rows.append({"record_type": "replicate_mean",
                                  "depth": int(r.depth), "ell": int(r.ell),
                                  "q": int(r.q), "estimator_size": int(size),
                                  "n_replicates": int(nrep),
                                  "estimate": float(m),
                                  "target": pop,
                                  "difference": float(m - pop),
                                  "standard_error": float(sem),
                                  "ci95_lo": float(m - tc * sem),
                                  "ci95_hi": float(m + tc * sem),
                                  "interval_kind": f"t({nrep-1})"})
            wide = sub.pivot(index="rep", columns="size",
                             values="omega_pred")
            if wide.shape[1] == 2:
                sizes = sorted(wide.columns)
                diff = wide[sizes[1]] - wide[sizes[0]]
                nrep = len(diff)
                sem = float(diff.std() / np.sqrt(nrep))
                tc = tcrit(nrep - 1)
                stat_rows.append({"record_type": "paired_size_shift",
                                  "depth": int(r.depth), "ell": int(r.ell),
                                  "q": int(r.q), "estimator_size": -1,
                                  "n_replicates": int(nrep),
                                  "estimate": float(diff.mean()),
                                  "target": 0.0,
                                  "difference": float(diff.mean()),
                                  "standard_error": sem,
                                  "ci95_lo": float(diff.mean() - tc * sem),
                                  "ci95_hi": float(diff.mean() + tc * sem),
                                  "interval_kind": f"paired t({nrep-1})"})
            hh = h2[(h2.depth == r.depth) & (h2.ell == r.ell)
                    & (h2.q == r.q)]
            if len(hh):
                oe = float(hh.om_extrap.iloc[0]) * (1.0 - GAMMA_PROBE)
                stat_rows.append({"record_type": "measured_extrapolation",
                                  "depth": int(r.depth), "ell": int(r.ell),
                                  "q": int(r.q), "estimator_size": 0,
                                  "n_replicates": 0,
                                  "estimate": oe, "target": pop,
                                  "difference": float(oe - pop),
                                  "standard_error": float("nan"),
                                  "ci95_lo": float("nan"),
                                  "ci95_hi": float("nan"),
                                  "interval_kind": "point"})
        pd.DataFrame(stat_rows).to_csv(out_csv_P2, index=False)
        for s_ in stat_rows:
            if s_["record_type"] == "replicate_mean":
                print(f"    ({s_['ell']},{s_['q']}) size "
                      f"{s_['estimator_size']:3d}: {s_['estimate']:.4f} "
                      f"vs pop {s_['target']:.4f}  "
                      f"CI [{s_['ci95_lo']:.4f}, {s_['ci95_hi']:.4f}] "
                      f"({s_['interval_kind']})")
            elif s_["record_type"] == "paired_size_shift":
                print(f"    ({s_['ell']},{s_['q']}) paired size shift "
                      f"{s_['estimate']:+.4f}, CI "
                      f"[{s_['ci95_lo']:+.4f}, {s_['ci95_hi']:+.4f}]")
    else:
        print("  [note] Phase H2 outputs not found; statistics deferred")
    if stat_rows:
        write_meta(out_csv_P2, "P-stats", inputs=[rep_p, h2_p])
    TIMINGS["P-stats"] = time.time() - t0s
    print(f"Phase P statistics complete in {TIMINGS['P-stats']:.1f}s "
          f"-> {out_csv_P2.name}")

# Worked example — auditing a device-style profile through the graph layer

A user-supplied dephasing weight profile is audited without touching the
density-matrix simulator: analytic rates, the machine-precision
difference-form certificate, protection structure and the support-cut
bound, with the finite-probe generator used only as the cross-check.


In [ ]:
# A device-style dephasing profile audited entirely through the graph layer:
# analytic rates, structural certificate, protection structure and the
# support-cut bound, without touching the density-matrix simulator.
device_w = np.array([1.8, 0.6, 1.1, 0.9, 1.4, 0.7, 1.2, 1.3])[:N]
device_ch = cac.InhomogeneousDephasing(device_w)
ana, pairs_w = cgg.analytic_rate_vector(device_ch, N, R)
cert = cgg.check_difference_form(device_ch, N, R)
L_w, _ = car.restricted_generator(device_ch, N, R, GAMMA_PROBE)
num_w = cgg.numeric_diag_rates(L_w)
mask_w = cgg.protected_mask(num_w)
rng = np.random.default_rng(EST_SEED)
th = rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
gvec = car.vec_offdiag(car.slot_modes(MODEL, th, BETA, 1, 0)[1], pairs_w)
bound, phi, rmin, lam = cgg.support_cut_bound(gvec, num_w, mask_w)
print("Worked example: device-style inhomogeneous dephasing profile")
print(f"  weights (mean-normalised): "
      f"{np.round(device_ch.weights, 3).tolist()}")
print(f"  difference-form certificate (should be ~1e-16): {cert:.2e}")
print(f"  numeric vs analytic worst dev: "
      f"{float(np.max(np.abs(num_w - ana))):.3e} "
      f"(probe bias at gamma_probe={GAMMA_PROBE:.0e})")
print(f"  rate range [{num_w.min():.3f}, {num_w.max():.3f}], "
      f"protected pairs {int(mask_w.sum())}")
print(f"  mode rate {lam:.4f} >= support-cut bound {bound:.4f} "
      f"(phi={phi:.3f}, r_min={rmin:.3f})")

# Outputs summary and run metadata


In [ ]:
print("=" * 76)
print("RUN SUMMARY")
print("=" * 76)
for src_name, src in (("cohalign_core.py", COHALIGN_CORE_SRC),
                      ("cohalign_rates.py", COHALIGN_RATES_SRC),
                      ("cohalign_bench.py", COHALIGN_BENCH_SRC),
                      ("cgl_graph.py", CGL_GRAPH_SRC)):
    (OUT_DIR / src_name).write_text(src)
outputs = sorted(p.name for p in OUT_DIR.glob("*.csv"))
figures = sorted(p.name for p in FIG_DIR.glob("*.png"))
print(f"MODE={MODE}  n={CFG['n']} L={CFG['L']} r={CFG['r']}")
print(f"CSV outputs ({len(outputs)}): {outputs}")
print(f"figures ({len(figures)}): {figures}")
for k, v in TIMINGS.items():
    print(f"  Phase {k}: {v:7.1f} s")
import hashlib, glob as _glob
nb_hash = "not_located (set CGL_NOTEBOOK_PATH to record it)"
_env_nb = os.environ.get("CGL_NOTEBOOK_PATH")
_cands = ([Path(_env_nb)] if _env_nb else []) +     [Path(p) for p in sorted(_glob.glob(str(ROOT / "coherence_graph_pipeline_v1_*.ipynb")), reverse=True)]
for cand in _cands:
    if cand.exists():
        nb_hash = hashlib.sha256(cand.read_bytes()).hexdigest()[:16]
        break
# backfill provenance sidecars for any CSV lacking one, honestly marked
for csv_p in sorted(OUT_DIR.glob("*.csv")):
    mp2 = csv_p.with_suffix(csv_p.suffix + ".meta.json")
    if not mp2.exists():
        # v1.4: a backfilled record carries no execution fingerprint at
        # all, so it can never be mistaken for execution-time provenance
        mp2.write_text(json.dumps({
            "phase": "backfill", "backfilled": True,
            "fingerprint_version": None, "config_fingerprint": None,
            "output_sha256":
                __import__("hashlib").sha256(csv_p.read_bytes()).hexdigest(),
            "note": "sidecar written after the fact; execution history "
                    "for this file is not documented",
            "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                         time.gmtime())}, indent=2))
        print(f"  [provenance] backfilled sidecar for {csv_p.name} "
              "(legacy status, permanent)")
if FIGURES_ONLY:
    _after = _csv_snapshot()
    changed = {k for k in set(_FIGURES_SNAPSHOT) | set(_after)
               if _FIGURES_SNAPSHOT.get(k) != _after.get(k)}
    if changed:
        raise RuntimeError("figures-only run modified numerical outputs: "
                           + ", ".join(sorted(changed)))
    print(f"[figures] integrity verified: {len(_after)} numerical CSVs "
          "unchanged by this rendering run")
manifest = {
    "pipeline": "coherence_graph_pipeline",
    "version": PIPELINE_VERSION,
    "mode": MODE, "config": {k: (list(v) if isinstance(v, tuple) else v)
                             for k, v in CFG.items()},
    "seeds": {"teacher": TEACHER_SEED, "preflight": PREFLIGHT_SEED,
              "est": EST_SEED, "meas": MEAS_SEED, "boot": BOOT_SEED,
              "sweep": SWEEP_SEED, "hess": HESS_SEED},
    "gamma_probe": GAMMA_PROBE, "delta_init": DELTA_INIT,
    "timings_seconds": {k: round(v, 2) for k, v in TIMINGS.items()},
    "total_wall_seconds": round(time.time() - t_notebook_start, 1),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "numpy": np.__version__, "pandas": pd.__version__,
    "notebook_sha256_16": nb_hash,
    "modules_from_cohalign_release": "v1.8.1 (verbatim)",
}
# merge with any existing manifest so run history accumulates instead of
# being overwritten (a v1.2 provenance repair)
mp = OUT_DIR / "MANIFEST.json"
history = []
if mp.exists():
    try:
        prev = json.loads(mp.read_text())
        history = prev.get("runs", [])
        if "mode" in prev and not history:
            history = [{"mode": prev.get("mode"),
                        "timings_seconds": prev.get("timings_seconds", {}),
                        "total_wall_seconds": prev.get("total_wall_seconds"),
                        "version": prev.get("version", "pre-1.2")}]
    except Exception:
        history = []
history.append({"mode": MODE, "version": PIPELINE_VERSION,
                "cache_only": len(TIMINGS) == 0,
                "phases_run": sorted(TIMINGS.keys()),
                "timings_seconds": {k: round(v, 2)
                                    for k, v in TIMINGS.items()},
                "total_wall_seconds": round(time.time() - t_notebook_start, 1),
                "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",
                                             time.gmtime())})
# per-file provenance summary: recomputed-this-session vs inherited
file_status = {}
for csv_p in sorted(OUT_DIR.glob("*.csv")):
    mp2 = csv_p.with_suffix(csv_p.suffix + ".meta.json")
    st = {"status": "no-sidecar"}
    if mp2.exists():
        try:
            mm = json.loads(mp2.read_text())
            if mm.get("backfilled"):
                st = {"status": "inherited (backfilled provenance)"}
            elif mm.get("fingerprint_version") == FINGERPRINT_VERSION and                     mm.get("phase_source_sha256"):
                st = {"status": "execution-bound",
                      "phase": mm.get("phase"),
                      "written_utc": mm.get("written_utc")}
            else:
                st = {"status": "inherited (legacy provenance)",
                      "phase": mm.get("phase")}
        except Exception:
            st = {"status": "unreadable sidecar"}
    file_status[csv_p.name] = st
manifest["file_provenance"] = file_status
n_exec = sum(1 for v in file_status.values()
             if v["status"] == "execution-bound")
print(f"file provenance: {n_exec} execution-bound, "
      f"{len(file_status) - n_exec} inherited or legacy "
      f"(itemised in MANIFEST.json)")
manifest["runs"] = history
mp.write_text(json.dumps(manifest, indent=2))
print(f"manifest -> {OUT_DIR / 'MANIFEST.json'}")
print(f"total wall time {manifest['total_wall_seconds']:.1f} s")

# Reproducibility, repository layout, and data availability

Executed top to bottom, this notebook writes to the results directory: the
phase CSVs with their provenance sidecars, the four module sources exactly
as executed (`cohalign_core.py`, `cohalign_rates.py`, `cohalign_bench.py`
and `cgl_graph.py`), and `MANIFEST.json` with the mode, configuration,
seeds, per-phase wall times and package versions. Publication figures are
rendered from these CSVs by the companion figures notebook.

Seeds are fixed throughout (teacher 42; preflight 11; estimator 1;
measurement 1234; bootstrap 7; sweep 33; Hessian 55), so smoke and paper
runs are exactly reproducible at their respective ensemble sizes. The
smoke mode exercises every code path at reduced ensembles and reduced
depth coverage; its physics comparisons are indicative only, and the
assertions are confined to Phase A's exact identities by design.
